# CatBoost regression for Singapore HDB monthly rent

This notebook builds a reproducible single-stage CatBoost model using the
team's `final_report` feature tables. It has four goals:

1. preserve the report's 149,999 valid training rows and all 50,000 official
   test IDs through reconstructed `ROW_ID` values;
2. use chronological validation that reflects future-rent prediction;
3. compare the same CatBoost specification with and without the report's six
   macro features; and
4. test both trend-residual and leakage-safe comparable-rent enhancements.

The report removes one training record whose lease commencement year (2022)
occurs after its rental approval year (2021). The notebook preserves that
original row's ID as an explicit audit item instead of silently renumbering
the remaining records. No test rows are removed.

## Repository layout

The notebook discovers `final_report` by walking upward from the current
working directory. It therefore runs from either the repository root or this
notebook's directory without editing paths. The expected input layout is:

```text
final_report/
  data/
    original_dataset/
      train.csv
      test.csv
    features/final/
      static_train_scaled.csv
      static_test_scaled.csv
      temporal_raw.csv
```

Model files and submissions are written to
`final_report/data/model_outputs/catboost/`. They are not written beside the
notebook.

Install the environment from the repository root before running:

```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
```


## Why CatBoost is appropriate

CatBoost is a strong fit for this tabular regression problem because the
report combines continuous measurements, one-hot features and high-cardinality
categorical location fields.

- **Native categorical handling:** town, block, street, planning area,
  station, school, and mall names can be passed directly as categories.
- **Ordered category statistics:** CatBoost constructs target-based category
  statistics in an ordered way, reducing the leakage risk of ordinary
  full-data target encoding.
- **Nonlinear interactions:** rent depends on combinations such as floor area,
  flat type, town, month and transport accessibility.
- **Compatible with the team table:** the report's one-hot and scaled numeric
  fields can be consumed directly; tree split order is unaffected by scaling.
- **Strong regularized baseline:** depth, learning rate, L2 regularization,
  stochastic sampling and early stopping are practical controls at this size.

A tree ensemble does not naturally extrapolate a rising trend beyond the
latest labelled month. The notebook therefore retains the temporal and
comparable-history experiments, but selects the final design using forward
validation rather than assuming extrapolation must help.


In [ ]:
from pathlib import Path
import json
import warnings

import catboost
from catboost import CatBoostRegressor, Pool
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


def find_final_report_root(start: Path) -> Path:
    # Locate the repository's final_report folder without fixed paths.
    start = start.resolve()
    checked = []
    for candidate in [start, *start.parents]:
        possible_roots = [candidate]
        if candidate.name != "final_report":
            possible_roots.append(candidate / "final_report")
        for root in possible_roots:
            root = root.resolve()
            if root in checked:
                continue
            checked.append(root)
            required = [
                root / "data" / "features" / "final"
                / "static_train_scaled.csv",
                root / "data" / "features" / "final"
                / "static_test_scaled.csv",
                root / "data" / "features" / "final"
                / "temporal_raw.csv",
                root / "data" / "original_dataset" / "train.csv",
                root / "data" / "original_dataset" / "test.csv",
            ]
            if all(path.is_file() for path in required):
                return root

    attempted = "\n- ".join(str(path) for path in checked)
    raise FileNotFoundError(
        "Could not locate final_report with the required data files. "
        "Run the notebook from the repository root or from a directory "
        "inside the repository, and store the data under final_report/data."
        f"\nChecked:\n- {attempted}"
    )


FINAL_REPORT_ROOT = find_final_report_root(Path.cwd())
PROJECT_ROOT = FINAL_REPORT_ROOT.parent
REPORT_DATA_ROOT = FINAL_REPORT_ROOT / "data"
REPORT_STATIC_ROOT = REPORT_DATA_ROOT / "features" / "final"
REPORT_ORIGINAL_ROOT = REPORT_DATA_ROOT / "original_dataset"
REPORT_TEMPORAL_PATH = REPORT_STATIC_ROOT / "temporal_raw.csv"
MODEL_ROOT = REPORT_DATA_ROOT / "model_outputs" / "catboost"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repository root:   {PROJECT_ROOT}")
print(f"Final report root: {FINAL_REPORT_ROOT}")
print(f"Model outputs:     {MODEL_ROOT}")
print(f"CatBoost:          {catboost.__version__}")


## 1. Load and verify the team's final-report feature tables

The report and our earlier bundle contain byte-identical raw competition
`train.csv` and `test.csv` files. Their processed representations differ:

- the report drops original training `ROW_ID=31739` because its lease begins
  after the recorded rental approval year;
- the report retains duplicated rental records and all 50,000 test rows;
- its categorical flat, school and DBSCAN fields are one-hot encoded;
- it does not include the earlier `diffusion_cluster` feature;
- its numerical scaler is fitted on January 2021 through March 2024 rather
  than on the entire labelled table; and
- `sample_type` is a pipeline split marker, so it is removed before modelling.

`ROW_ID`, `FLAT_TYPE` and `FLAT_MODEL` are reconstructed from the report's
own raw files. The latter two are helper fields for diagnostics and
comparable-rent grouping; their one-hot report columns remain the base model
representation. Assertions stop execution if row order or target alignment
no longer matches the supplied report.


In [ ]:
TARGET = "MONTHLY_RENT"
REPORT_SPLIT_COLUMN = "sample_type"

report_train = pd.read_csv(REPORT_STATIC_ROOT / "static_train_scaled.csv")
report_test = pd.read_csv(REPORT_STATIC_ROOT / "static_test_scaled.csv")

raw_train_reference = pd.read_csv(
    REPORT_ORIGINAL_ROOT / "train.csv",
    usecols=[
        "RENT_APPROVAL_DATE",
        "LEASE_COMMENCE_DATE",
        "FLAT_TYPE",
        "FLAT_MODEL",
        TARGET,
    ],
)
raw_test_reference = pd.read_csv(
    REPORT_ORIGINAL_ROOT / "test.csv",
    usecols=[
        "RENT_APPROVAL_DATE",
        "LEASE_COMMENCE_DATE",
        "FLAT_TYPE",
        "FLAT_MODEL",
    ],
)
raw_train_reference.insert(0, "ROW_ID", np.arange(len(raw_train_reference)))
raw_test_reference.insert(0, "ROW_ID", np.arange(len(raw_test_reference)))

train_approval_year = raw_train_reference[
    "RENT_APPROVAL_DATE"
].str[:4].astype(int)
valid_lease = (
    train_approval_year
    >= raw_train_reference["LEASE_COMMENCE_DATE"].astype(int)
)
removed_report_rows = raw_train_reference.loc[
    ~valid_lease,
    [
        "ROW_ID",
        "RENT_APPROVAL_DATE",
        "LEASE_COMMENCE_DATE",
        "FLAT_TYPE",
        "FLAT_MODEL",
        TARGET,
    ],
].copy()
raw_train_reference = raw_train_reference.loc[valid_lease].reset_index(drop=True)

assert len(report_train) == len(raw_train_reference) == 149_999
assert len(report_test) == len(raw_test_reference) == 50_000
assert removed_report_rows["ROW_ID"].tolist() == [31_739]
assert report_train[TARGET].equals(raw_train_reference[TARGET])

report_train_month = (
    report_train["RENT_APPROVAL_YEAR"].astype(int).astype(str)
    + "-"
    + report_train["RENT_APPROVAL_MONTH"].astype(int).astype(str).str.zfill(2)
)
report_test_month = (
    report_test["RENT_APPROVAL_YEAR"].astype(int).astype(str)
    + "-"
    + report_test["RENT_APPROVAL_MONTH"].astype(int).astype(str).str.zfill(2)
)
assert report_train_month.equals(raw_train_reference["RENT_APPROVAL_DATE"])
assert report_test_month.equals(raw_test_reference["RENT_APPROVAL_DATE"])

train_raw = report_train.drop(columns=[REPORT_SPLIT_COLUMN]).copy()
test_raw = report_test.drop(columns=[REPORT_SPLIT_COLUMN]).copy()
train_raw.insert(0, "ROW_ID", raw_train_reference["ROW_ID"].to_numpy())
test_raw.insert(0, "ROW_ID", raw_test_reference["ROW_ID"].to_numpy())

for frame, reference in [
    (train_raw, raw_train_reference),
    (test_raw, raw_test_reference),
]:
    frame["FLAT_TYPE"] = (
        reference["FLAT_TYPE"]
        .astype("string")
        .str.replace("-", " ", regex=False)
        .str.lower()
    )
    frame["FLAT_MODEL"] = (
        reference["FLAT_MODEL"].astype("string").str.lower()
    )

assert train_raw["ROW_ID"].is_unique
assert test_raw["ROW_ID"].equals(
    pd.Series(np.arange(len(test_raw)), name="ROW_ID")
)
assert [column for column in train_raw if column != TARGET] == list(
    test_raw.columns
)
assert train_raw[TARGET].notna().all()
assert not train_raw.isna().any().any()
assert not test_raw.isna().any().any()

display(pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(train_raw), len(test_raw)],
    "columns": [train_raw.shape[1], test_raw.shape[1]],
    "missing_cells": [
        int(train_raw.isna().sum().sum()),
        int(test_raw.isna().sum().sum()),
    ],
    "unique_ROW_ID": [
        train_raw["ROW_ID"].is_unique,
        test_raw["ROW_ID"].is_unique,
    ],
}))
print("Report preprocessing removed this invalid training record:")
display(removed_report_rows)


## 2. Deterministic feature engineering

Only values available from the row's approval month and building dates are
used:

- `RENT_APPROVAL_DATE` is a split/control field, not a predictor.
- `MONTH_INDEX` supplies a continuous market-time index.
- `MONTH_SIN` and `MONTH_COS` represent annual seasonality.
- `LEASE_AGE` and `BUILDING_AGE` are interpretable age measurements.
- `RENT_MONTH_CAT` lets CatBoost learn month-specific categorical effects.

The report's numerical columns are already standardized using its
January-2021-to-March-2024 training window and are not scaled again. Its `-1`
missing-value sentinel is retained for exact team compatibility. Reconstructed
`FLAT_TYPE` and `FLAT_MODEL` are helper keys, not duplicate base predictors.


In [ ]:
def add_deterministic_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["RENT_APPROVAL_DATE"] = pd.to_datetime(
        dict(
            year=out["RENT_APPROVAL_YEAR"].astype(int),
            month=out["RENT_APPROVAL_MONTH"].astype(int),
            day=1,
        )
    )
    out["MONTH_INDEX"] = (
        (out["RENT_APPROVAL_YEAR"].astype(int) - 2021) * 12
        + out["RENT_APPROVAL_MONTH"].astype(int)
        - 1
    )
    angle = 2 * np.pi * out["RENT_APPROVAL_MONTH"].astype(float) / 12
    out["MONTH_SIN"] = np.sin(angle)
    out["MONTH_COS"] = np.cos(angle)
    out["LEASE_AGE"] = (
        out["RENT_APPROVAL_YEAR"].astype(float)
        - out["LEASE_COMMENCE_DATE"].astype(float)
    )
    out["BUILDING_AGE"] = (
        out["RENT_APPROVAL_YEAR"].astype(float)
        - out["YEAR_COMPLETED"].astype(float)
    )
    out.loc[out["LEASE_AGE"] < 0, "LEASE_AGE"] = np.nan
    out.loc[out["BUILDING_AGE"] < 0, "BUILDING_AGE"] = np.nan
    out["RENT_MONTH_CAT"] = out["RENT_APPROVAL_MONTH"].astype("string")
    return out


train = add_deterministic_features(train_raw)
test = add_deterministic_features(test_raw)

display(pd.DataFrame({
    "split": ["train", "test"],
    "first_month": [train["RENT_APPROVAL_DATE"].min(), test["RENT_APPROVAL_DATE"].min()],
    "last_month": [train["RENT_APPROVAL_DATE"].max(), test["RENT_APPROVAL_DATE"].max()],
    "distinct_months": [train["RENT_APPROVAL_DATE"].nunique(), test["RENT_APPROVAL_DATE"].nunique()],
}))

## 3. Join the report's macro panel, but quarantine realized future features

The report's temporal table has one row per month from January 2021 through
July 2026. It contains six macro variables, 159 stock-market fields and 12
COE fields. Stock and COE values are calculated from the same rental month;
they are therefore known only after or during that month and are unsafe for
a single fixed-origin forecast of the entire future test horizon.

The six macro variables are retained for the existing validation ablation,
but the final fixed-origin model continues to exclude them unless genuine
origin-date forecasts are supplied. Same-month stock and COE fields are not
joined to row-level model data by default. The report's two `sample_type`
columns are pipeline markers rather than predictors and are ignored.


In [ ]:
temporal_report = pd.read_csv(REPORT_TEMPORAL_PATH)
temporal_report["RENT_APPROVAL_DATE"] = pd.to_datetime(
    dict(
        year=temporal_report["RENT_APPROVAL_YEAR"].astype(int),
        month=temporal_report["RENT_APPROVAL_MONTH"].astype(int),
        day=1,
    )
)
assert temporal_report["RENT_APPROVAL_DATE"].is_unique

MACRO_COLUMNS = [
    column for column in temporal_report if column.startswith("MACRO_")
]
SAME_MONTH_FUTURE_RISK_COLUMNS = [
    column
    for column in temporal_report
    if column.startswith("stk_") or column.startswith("coe_")
]
assert len(MACRO_COLUMNS) == 6
assert len(SAME_MONTH_FUTURE_RISK_COLUMNS) == 171

macro = temporal_report[["RENT_APPROVAL_DATE", *MACRO_COLUMNS]].copy()


def join_macro(frame: pd.DataFrame) -> pd.DataFrame:
    original_rows = len(frame)
    joined = frame.merge(
        macro,
        on="RENT_APPROVAL_DATE",
        how="left",
        validate="many_to_one",
        sort=False,
    )
    assert len(joined) == original_rows
    assert joined[MACRO_COLUMNS].notna().all().all()
    return joined


train = join_macro(train)
test = join_macro(test)

display(macro.set_index("RENT_APPROVAL_DATE").loc[
    train["RENT_APPROVAL_DATE"].min():test["RENT_APPROVAL_DATE"].max()
].tail())
print(f"Macro columns joined for ablation: {len(MACRO_COLUMNS)}")
print(
    "Same-month stock/COE columns quarantined from fixed-origin models: "
    f"{len(SAME_MONTH_FUTURE_RISK_COLUMNS)}"
)


## 4. Prepare CatBoost predictors

The report's one-hot columns are retained as numeric predictors. Its
high-cardinality identifier fields, including town, block, street, planning
area and nearby-place names, are passed to CatBoost as categoricals. Fields
designated categorical by the team's feature configuration are also cast to
strings.

`FLAT_TYPE` and `FLAT_MODEL` are reconstructed only for diagnostics and
comparable-history grouping, because the report already supplies their
one-hot columns. `ROW_ID`, `sample_type` and the full date are never model
predictors.

Data are sorted by month and fitted with `has_time=True` so CatBoost preserves
chronological order when constructing ordered category statistics.


In [ ]:
EXPLICIT_CATEGORICALS = [
    "POSTAL_CODE",
    "MAX_FLOOR",
    "mrt_traffic_busiest_hour",
    "train_lines",
    "station_is_open",
    "RENT_MONTH_CAT",
]
REPORT_HELPER_ONLY_COLUMNS = {"FLAT_TYPE", "FLAT_MODEL"}
NON_PREDICTORS = {
    "ROW_ID",
    TARGET,
    "RENT_APPROVAL_DATE",
    REPORT_SPLIT_COLUMN,
    *REPORT_HELPER_ONLY_COLUMNS,
}


def make_feature_frame(frame: pd.DataFrame, include_macro: bool):
    excluded = set(NON_PREDICTORS)
    if not include_macro:
        excluded.update(MACRO_COLUMNS)
    feature_columns = [
        column for column in frame.columns if column not in excluded
    ]
    X = frame[feature_columns].copy()
    categorical_columns = [
        column
        for column in feature_columns
        if pd.api.types.is_object_dtype(X[column])
        or isinstance(X[column].dtype, pd.StringDtype)
        or column in EXPLICIT_CATEGORICALS
    ]
    for column in categorical_columns:
        X[column] = X[column].fillna("__MISSING__").astype(str)
    numeric_columns = [
        column for column in feature_columns
        if column not in categorical_columns
    ]
    X[numeric_columns] = X[numeric_columns].replace(
        [np.inf, -np.inf], np.nan
    )
    return X, categorical_columns


X_check, categorical_check = make_feature_frame(
    train,
    include_macro=False,
)
display(pd.DataFrame({
    "predictor_columns": [X_check.shape[1]],
    "categorical_columns": [len(categorical_check)],
    "numeric_columns": [
        X_check.shape[1] - len(categorical_check)
    ],
}))
print("Categorical columns:")
print(categorical_check)


## 5. Chronological holdout

The final 12 distinct training months are held out. All earlier months form the
fit set. This tests performance on a genuinely later period and keeps duplicate
records from the same approval month in one partition.

MAE is easy to interpret in Singapore dollars. RMSE penalizes large errors more
strongly and is used for early stopping. Monthly bias is also reported because a
model can have acceptable row-level error while systematically underpredicting
an upward market.

In [ ]:
VALIDATION_MONTHS = 12
ordered_months = np.array(sorted(train["RENT_APPROVAL_DATE"].unique()))
validation_start = pd.Timestamp(ordered_months[-VALIDATION_MONTHS])

fit_data = train.loc[train["RENT_APPROVAL_DATE"] < validation_start].copy()
valid_data = train.loc[train["RENT_APPROVAL_DATE"] >= validation_start].copy()
fit_data = fit_data.sort_values(["RENT_APPROVAL_DATE", "ROW_ID"]).reset_index(drop=True)
valid_data = valid_data.sort_values(["RENT_APPROVAL_DATE", "ROW_ID"]).reset_index(drop=True)

assert fit_data["RENT_APPROVAL_DATE"].max() < valid_data["RENT_APPROVAL_DATE"].min()

display(pd.DataFrame({
    "partition": ["fit", "validation"],
    "rows": [len(fit_data), len(valid_data)],
    "first_month": [fit_data["RENT_APPROVAL_DATE"].min(), valid_data["RENT_APPROVAL_DATE"].min()],
    "last_month": [fit_data["RENT_APPROVAL_DATE"].max(), valid_data["RENT_APPROVAL_DATE"].max()],
}))

In [ ]:
def regression_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "R2": r2_score(actual, predicted),
        "BIAS": float(np.mean(predicted - actual)),
    }


def fit_monthly_trend(frame: pd.DataFrame):
    monthly = (
        frame.groupby("MONTH_INDEX", as_index=False)[TARGET]
        .median()
        .sort_values("MONTH_INDEX")
    )
    model = LinearRegression()
    model.fit(monthly[["MONTH_INDEX"]], monthly[TARGET])
    return model, monthly


trend_only_model, fit_monthly_medians = fit_monthly_trend(fit_data)
median_prediction = np.repeat(fit_data[TARGET].median(), len(valid_data))
trend_prediction = trend_only_model.predict(valid_data[["MONTH_INDEX"]])

results = [
    {"experiment": "Global median", "include_macro": False, "target_mode": "baseline", **regression_metrics(valid_data[TARGET], median_prediction)},
    {"experiment": "Linear month trend", "include_macro": False, "target_mode": "baseline", **regression_metrics(valid_data[TARGET], trend_prediction)},
]

display(pd.DataFrame(results).set_index("experiment"))

## 6. CatBoost experiments

Four comparable models are fitted:

1. direct rent prediction without macro variables;
2. direct rent prediction with the macro group;
3. linear market trend plus CatBoost residuals, without macro variables; and
4. the same trend-residual model with macro variables.

Early stopping selects the useful number of trees on the chronological holdout.
The same regularized parameters are used across experiments so differences are
attributable to target design and the macro feature group.

In [ ]:
CATBOOST_PARAMS = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "iterations": 1_000,
    "learning_rate": 0.05,
    "depth": 8,
    "l2_leaf_reg": 8.0,
    "random_strength": 0.5,
    "bagging_temperature": 0.5,
    "random_seed": RANDOM_SEED,
    "has_time": True,
    "allow_writing_files": False,
    "thread_count": -1,
    "verbose": 200,
}

experiment_models = {}
experiment_predictions = {}
experiment_trends = {}


def run_catboost_experiment(name: str, include_macro: bool, target_mode: str):
    X_fit, cat_columns = make_feature_frame(fit_data, include_macro=include_macro)
    X_valid, valid_cat_columns = make_feature_frame(valid_data, include_macro=include_macro)
    assert list(X_fit.columns) == list(X_valid.columns)
    assert cat_columns == valid_cat_columns

    trend_model = None
    if target_mode == "raw":
        y_fit = fit_data[TARGET].to_numpy(dtype=float)
        y_valid = valid_data[TARGET].to_numpy(dtype=float)
    elif target_mode == "trend_residual":
        trend_model, _ = fit_monthly_trend(fit_data)
        fit_trend = trend_model.predict(fit_data[["MONTH_INDEX"]])
        valid_trend = trend_model.predict(valid_data[["MONTH_INDEX"]])
        y_fit = fit_data[TARGET].to_numpy(dtype=float) - fit_trend
        y_valid = valid_data[TARGET].to_numpy(dtype=float) - valid_trend
    else:
        raise ValueError(target_mode)

    fit_pool = Pool(X_fit, label=y_fit, cat_features=cat_columns)
    valid_pool = Pool(X_valid, label=y_valid, cat_features=cat_columns)
    model = CatBoostRegressor(**CATBOOST_PARAMS)
    model.fit(
        fit_pool,
        eval_set=valid_pool,
        use_best_model=True,
        early_stopping_rounds=100,
    )

    prediction = model.predict(X_valid)
    if trend_model is not None:
        prediction = prediction + trend_model.predict(valid_data[["MONTH_INDEX"]])

    metrics = regression_metrics(valid_data[TARGET], prediction)
    results.append({
        "experiment": name,
        "include_macro": include_macro,
        "target_mode": target_mode,
        "best_iteration": int(model.get_best_iteration() + 1),
        **metrics,
    })
    experiment_models[name] = model
    experiment_predictions[name] = prediction
    experiment_trends[name] = trend_model
    print(name, metrics)


run_catboost_experiment("CatBoost raw", include_macro=False, target_mode="raw")
run_catboost_experiment("CatBoost raw + macro", include_macro=True, target_mode="raw")
run_catboost_experiment("Trend + CatBoost residual", include_macro=False, target_mode="trend_residual")
run_catboost_experiment("Trend + CatBoost residual + macro", include_macro=True, target_mode="trend_residual")

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
results_df["RMSE_GAIN_VS_RAW_CATBOOST"] = (
    results_df.loc[results_df["experiment"].eq("CatBoost raw"), "RMSE"].iloc[0]
    - results_df["RMSE"]
)
display(results_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(data=results_df, y="experiment", x="RMSE", ax=axes[0], color="#4C78A8")
axes[0].set_title("Chronological validation RMSE")
axes[0].set_xlabel("RMSE (SGD, lower is better)")
axes[0].set_ylabel("")
sns.barplot(data=results_df, y="experiment", x="MAE", ax=axes[1], color="#59A14F")
axes[1].set_title("Chronological validation MAE")
axes[1].set_xlabel("MAE (SGD, lower is better)")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

raw_rmse = results_df.loc[results_df["experiment"].eq("CatBoost raw"), "RMSE"].iloc[0]
raw_macro_rmse = results_df.loc[results_df["experiment"].eq("CatBoost raw + macro"), "RMSE"].iloc[0]
trend_rmse = results_df.loc[results_df["experiment"].eq("Trend + CatBoost residual"), "RMSE"].iloc[0]
trend_macro_rmse = results_df.loc[
    results_df["experiment"].eq("Trend + CatBoost residual + macro"), "RMSE"
].iloc[0]

print(f"Macro RMSE change for raw target:       {raw_rmse - raw_macro_rmse:,.2f} SGD")
print(f"Macro RMSE change for trend residuals:  {trend_rmse - trend_macro_rmse:,.2f} SGD")
print("Positive means macro improved the holdout.")

### Interpreting the macro comparison

A lower error with macro features on this holdout shows potential incremental
signal, but it does not by itself authorize realized future macro data in the
test set. A robust decision should require improvement across several forward
folds. For a fixed-origin submission, either exclude macro variables, freeze
them at the origin, or replace future values with forecasts produced using only
information available at the origin.

## 7. Diagnostics for the best fixed-origin-safe model

The final fixed-origin model is selected only from the two non-macro CatBoost
experiments. This avoids using realized macro observations from beyond the
training cutoff. Diagnostics are calculated on the untouched chronological
holdout.

In [ ]:
safe_candidates = results_df.loc[
    results_df["experiment"].isin(["CatBoost raw", "Trend + CatBoost residual"])
]
BEST_SAFE_EXPERIMENT = safe_candidates.sort_values("RMSE").iloc[0]["experiment"]
best_validation_model = experiment_models[BEST_SAFE_EXPERIMENT]
best_validation_prediction = experiment_predictions[BEST_SAFE_EXPERIMENT]

diagnostics = valid_data[["ROW_ID", "RENT_APPROVAL_DATE", "TOWN", "FLAT_TYPE", TARGET]].copy()
diagnostics["PREDICTION"] = best_validation_prediction
diagnostics["ERROR"] = diagnostics["PREDICTION"] - diagnostics[TARGET]
diagnostics["ABS_ERROR"] = diagnostics["ERROR"].abs()

monthly_diagnostics = diagnostics.groupby("RENT_APPROVAL_DATE").agg(
    rows=("ROW_ID", "size"),
    actual_mean=(TARGET, "mean"),
    prediction_mean=("PREDICTION", "mean"),
    MAE=("ABS_ERROR", "mean"),
    bias=("ERROR", "mean"),
).reset_index()

print(f"Best fixed-origin-safe experiment: {BEST_SAFE_EXPERIMENT}")
display(monthly_diagnostics)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(monthly_diagnostics["RENT_APPROVAL_DATE"], monthly_diagnostics["actual_mean"], marker="o", label="Actual")
axes[0].plot(monthly_diagnostics["RENT_APPROVAL_DATE"], monthly_diagnostics["prediction_mean"], marker="o", label="Predicted")
axes[0].set_title("Monthly mean rent")
axes[0].set_ylabel("SGD")
axes[0].legend()
axes[1].bar(monthly_diagnostics["RENT_APPROVAL_DATE"], monthly_diagnostics["bias"], width=20, color="#E15759")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Monthly prediction bias")
axes[1].set_ylabel("Predicted − actual (SGD)")
plt.tight_layout()
plt.show()

In [ ]:
best_include_macro = False
X_fit_best, best_cat_columns = make_feature_frame(fit_data, include_macro=best_include_macro)
feature_importance = pd.DataFrame({
    "feature": X_fit_best.columns,
    "importance": best_validation_model.get_feature_importance(),
}).sort_values("importance", ascending=False)

display(feature_importance.head(25))
top = feature_importance.head(25).sort_values("importance")
plt.figure(figsize=(9, 8))
plt.barh(top["feature"], top["importance"], color="#4C78A8")
plt.title(f"CatBoost feature importance: {BEST_SAFE_EXPERIMENT}")
plt.xlabel("PredictionValuesChange importance")
plt.tight_layout()
plt.show()

print(
    "Importance is associative, not causal. Correlated geography and amenity "
    "features can divide or substitute for one another's importance."
)

## 8. Final evaluation: model performance and forecast windows

The notebook uses three different time windows, and only one of them has been
used to calculate the performance metrics below:

- **Model-fitting window:** January 2021 to March 2024.
- **Held-out validation window:** April 2024 to March 2025. These labelled rows
  were not used for fitting or early-stage feature construction. All reported
  MAE, RMSE, R² and bias values are calculated on this window.
- **Official unlabelled test window:** March 2025 to July 2026. Its rental
  labels are unavailable, so genuine test error cannot yet be calculated.

The dates are therefore mostly, but not completely, separated: March 2025 is
present in both the labelled data and the official test feature table. The rows
are different, but the shared calendar month should be stated explicitly. The
held-out validation period is the test-like window used for model selection; it
must not be described as performance on the official competition test set.

In [ ]:
import matplotlib.dates as mdates

window_summary = pd.DataFrame([
    {
        'window': 'Model fitting',
        'start': fit_data['RENT_APPROVAL_DATE'].min(),
        'end': fit_data['RENT_APPROVAL_DATE'].max(),
        'rows': len(fit_data),
        'labels_available': True,
        'purpose': 'Fit models and categorical statistics',
    },
    {
        'window': 'Held-out validation',
        'start': valid_data['RENT_APPROVAL_DATE'].min(),
        'end': valid_data['RENT_APPROVAL_DATE'].max(),
        'rows': len(valid_data),
        'labels_available': True,
        'purpose': 'Model selection and all reported error metrics',
    },
    {
        'window': 'Official test',
        'start': test['RENT_APPROVAL_DATE'].min(),
        'end': test['RENT_APPROVAL_DATE'].max(),
        'rows': len(test),
        'labels_available': False,
        'purpose': 'Generate submission predictions only',
    },
])
display(window_summary)

fig, ax = plt.subplots(figsize=(13, 3.2))
timeline_colours = {
    'Model fitting': '#4C78A8',
    'Held-out validation': '#E15759',
    'Official test': '#59A14F',
}
for y, row in window_summary.reset_index(drop=True).iterrows():
    ax.plot(
        [row['start'], row['end']],
        [y, y],
        linewidth=18,
        solid_capstyle='butt',
        color=timeline_colours[row['window']],
        label=row['window'],
    )
    ax.scatter([row['start'], row['end']], [y, y], color=timeline_colours[row['window']], s=45)
ax.axvline(pd.Timestamp('2025-03-01'), color='black', linestyle='--', linewidth=1)
ax.annotate(
    'March 2025 appears in both labelled and official test date ranges',
    xy=(pd.Timestamp('2025-03-01'), 1.5),
    xytext=(pd.Timestamp('2022-10-01'), 2.45),
    arrowprops={'arrowstyle': '->', 'color': 'black'},
    fontsize=9,
)
ax.set_yticks(range(len(window_summary)))
ax.set_yticklabels(window_summary['window'])
ax.set_ylim(-0.6, 2.7)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_title('Training, chronological validation, and official test windows')
ax.set_xlabel('Rental approval month')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
final_evaluation = results_df.copy()
global_median_rmse = final_evaluation.loc[
    final_evaluation['experiment'].eq('Global median'), 'RMSE'
].iloc[0]
global_median_mae = final_evaluation.loc[
    final_evaluation['experiment'].eq('Global median'), 'MAE'
].iloc[0]
final_evaluation['RMSE_RANK'] = final_evaluation['RMSE'].rank(method='min').astype(int)
final_evaluation['RMSE_IMPROVEMENT_VS_MEDIAN_PCT'] = (
    100 * (global_median_rmse - final_evaluation['RMSE']) / global_median_rmse
)
final_evaluation['MAE_IMPROVEMENT_VS_MEDIAN_PCT'] = (
    100 * (global_median_mae - final_evaluation['MAE']) / global_median_mae
)
final_evaluation = final_evaluation.sort_values(['RMSE_RANK', 'MAE']).reset_index(drop=True)
display(
    final_evaluation[[
        'RMSE_RANK', 'experiment', 'MAE', 'RMSE', 'R2', 'BIAS',
        'RMSE_IMPROVEMENT_VS_MEDIAN_PCT', 'MAE_IMPROVEMENT_VS_MEDIAN_PCT',
    ]].style.format({
        'MAE': '{:,.1f}',
        'RMSE': '{:,.1f}',
        'R2': '{:.3f}',
        'BIAS': '{:+,.1f}',
        'RMSE_IMPROVEMENT_VS_MEDIAN_PCT': '{:+.1f}%',
        'MAE_IMPROVEMENT_VS_MEDIAN_PCT': '{:+.1f}%',
    })
)

In [ ]:
validation_prediction_map = {
    'Global median': median_prediction,
    'Linear month trend': trend_prediction,
    **experiment_predictions,
}
expected_models = set(results_df['experiment'])
assert set(validation_prediction_map) == expected_models

actual_monthly = (
    valid_data.groupby('RENT_APPROVAL_DATE', as_index=False)[TARGET]
    .mean()
    .rename(columns={TARGET: 'ACTUAL_MEAN'})
)
monthly_prediction_frames = []
for model_name, prediction in validation_prediction_map.items():
    frame = valid_data[['RENT_APPROVAL_DATE']].copy()
    frame['PREDICTION'] = np.asarray(prediction, dtype=float)
    frame = frame.groupby('RENT_APPROVAL_DATE', as_index=False)['PREDICTION'].mean()
    frame['MODEL'] = model_name
    monthly_prediction_frames.append(frame)
monthly_model_predictions = pd.concat(monthly_prediction_frames, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
baseline_models = ['Global median', 'Linear month trend']
catboost_models = [
    'CatBoost raw',
    'CatBoost raw + macro',
    'Trend + CatBoost residual',
    'Trend + CatBoost residual + macro',
]
for ax, model_names, title in zip(
    axes,
    [baseline_models, catboost_models],
    ['Baseline predictions', 'CatBoost experiment predictions'],
):
    ax.plot(
        actual_monthly['RENT_APPROVAL_DATE'],
        actual_monthly['ACTUAL_MEAN'],
        color='black', linewidth=3, marker='o', label='Actual mean rent',
    )
    for model_name in model_names:
        subset = monthly_model_predictions.loc[monthly_model_predictions['MODEL'].eq(model_name)]
        ax.plot(
            subset['RENT_APPROVAL_DATE'], subset['PREDICTION'],
            linewidth=1.8, marker='.', label=model_name,
        )
    ax.set_title(f"{title}\nHeld-out validation: Apr 2024–Mar 2025")
    ax.set_xlabel('Rental approval month')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
axes[0].set_ylabel('Mean monthly rent (SGD)')
plt.tight_layout()
plt.show()

In [ ]:
best_metrics = results_df.loc[results_df['experiment'].eq(BEST_SAFE_EXPERIMENT)].iloc[0]
plot_actual = valid_data[TARGET].to_numpy(dtype=float)
plot_prediction = np.asarray(best_validation_prediction, dtype=float)
plot_min = min(plot_actual.min(), plot_prediction.min())
plot_max = max(plot_actual.max(), plot_prediction.max())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hexbin = axes[0].hexbin(
    plot_actual, plot_prediction, gridsize=55, mincnt=1, cmap='viridis',
)
axes[0].plot([plot_min, plot_max], [plot_min, plot_max], '--', color='red', linewidth=1.5)
axes[0].set_xlim(plot_min, plot_max)
axes[0].set_ylim(plot_min, plot_max)
axes[0].set_xlabel('Actual monthly rent (SGD)')
axes[0].set_ylabel('Predicted monthly rent (SGD)')
axes[0].set_title(f"{BEST_SAFE_EXPERIMENT}: actual vs predicted\nApr 2024–Mar 2025")
axes[0].text(
    0.03, 0.97,
    f"MAE = {best_metrics['MAE']:,.1f}\nRMSE = {best_metrics['RMSE']:,.1f}\nR² = {best_metrics['R2']:.3f}",
    transform=axes[0].transAxes, va='top',
    bbox={'facecolor': 'white', 'alpha': 0.85, 'edgecolor': 'none'},
)
fig.colorbar(hexbin, ax=axes[0], label='Validation rows per hexagon')

bias_colours = np.where(monthly_diagnostics['bias'] >= 0, '#4C78A8', '#E15759')
axes[1].bar(
    monthly_diagnostics['RENT_APPROVAL_DATE'],
    monthly_diagnostics['bias'],
    width=20, color=bias_colours,
)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title(f"{BEST_SAFE_EXPERIMENT}: monthly bias\nPredicted minus actual")
axes[1].set_xlabel('Rental approval month')
axes[1].set_ylabel('Bias (SGD)')
axes[1].grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

### Interpretation and final conclusion

| Rank | Model | MAE | RMSE | R² | Bias |
|---:|---|---:|---:|---:|---:|
| 1 | CatBoost raw | 343.3 | 469.7 | 0.492 | −1.8 |
| 2 | CatBoost raw + macro | 349.3 | 472.6 | 0.486 | −32.8 |
| 3 | Trend + CatBoost residual | 379.6 | 525.6 | 0.364 | +215.1 |
| 4 | Trend + CatBoost residual + macro | 402.7 | 550.6 | 0.302 | +253.0 |
| 5 | Linear month trend | 608.2 | 773.3 | −0.376 | +388.2 |
| 6 | Global median | 752.2 | 909.0 | −0.902 | −625.9 |

The results support the following interpretation:

1. **The global median substantially underpredicts later rents.** The fitting
   period contains many lower-rent observations from 2021–2023, while the
   validation period is later and more expensive. This produces a large
   negative bias.
2. **A straight-line trend overcorrects.** Rental growth was not linear across
   the full period. Extrapolating one slope from earlier monthly medians into
   2024–2025 produces large positive bias.
3. **Direct CatBoost performs best because property and location interactions
   explain substantial cross-sectional variation.** Flat type, floor area,
   postal code, planning area, building age, amenities and the month index can
   interact without a separately imposed linear market path.
4. **The six macro variables do not add stable incremental signal in this
   holdout.** Although there are 114,842 fitting rows, there are only 39 distinct
   fitting months. Every row in a month receives the same macro values. These
   variables also overlap strongly with `MONTH_INDEX`, and their apparent level
   correlations can reflect a shared trend rather than a persistent forecasting
   relationship. Raw-target RMSE therefore worsens by about $2.92 with macro
   variables.
5. **The trend-residual models inherit the error of the extrapolated trend.**
   CatBoost learns residual patterns from the fitting period, but it cannot fully
   offset an incorrectly extrapolated market component in later months. Adding
   macro variables introduces further correlated time signals and worsens this
   design again.
6. **An R² near 0.49 is useful but leaves substantial unexplained variation.**
   The table lacks unit-level identifiers, exact floor level, interior condition,
   renovation status and detailed tenancy terms. These omissions, together with
   negotiation noise, limit row-level accuracy even when broad property and
   location characteristics are known.
7. **Near-zero aggregate bias hides a recent extrapolation problem.** The selected
   model moves from modest overprediction in early validation months to roughly
   $62 underprediction in March 2025. Tree models interpolate well inside their
   observed range but do not naturally extend a changing market trend beyond the
   latest fitting months.

**Selection decision:** `CatBoost raw` remains the final fixed-origin-safe model.
It has the lowest chronological RMSE, the lowest MAE, the highest R² and almost
zero overall bias. It is refitted on all 149,999 report-aligned labelled rows using the 508
iterations selected by early stopping. Macro features remain an experimental
group for rolling-origin or forecast-supported evaluation, not inputs to the
current fixed-origin submission.

## 9. Train the final fixed-origin model and create predictions

The best non-macro target design is refitted on all labeled months. The selected
validation iteration count is reused instead of running early stopping against
the training data. If the trend-residual design wins, its linear trend is refit
on all observed monthly medians and extrapolated into the test period.

Predictions are clipped at zero only as a physical safeguard. `ROW_ID` restores
the exact 50,000-row submission order.

In [ ]:
best_row = results_df.loc[results_df["experiment"].eq(BEST_SAFE_EXPERIMENT)].iloc[0]
best_target_mode = best_row["target_mode"]
best_iterations = int(best_row.get("best_iteration", CATBOOST_PARAMS["iterations"]))
best_iterations = max(best_iterations, 50)

full_train = train.sort_values(["RENT_APPROVAL_DATE", "ROW_ID"]).reset_index(drop=True)
ordered_test = test.sort_values("ROW_ID").reset_index(drop=True)
X_full, full_cat_columns = make_feature_frame(full_train, include_macro=False)
X_test, test_cat_columns = make_feature_frame(ordered_test, include_macro=False)
assert list(X_full.columns) == list(X_test.columns)
assert full_cat_columns == test_cat_columns

final_trend_model = None
if best_target_mode == "raw":
    final_target = full_train[TARGET].to_numpy(dtype=float)
elif best_target_mode == "trend_residual":
    final_trend_model, final_monthly_medians = fit_monthly_trend(full_train)
    final_train_trend = final_trend_model.predict(full_train[["MONTH_INDEX"]])
    final_target = full_train[TARGET].to_numpy(dtype=float) - final_train_trend
else:
    raise ValueError(best_target_mode)

final_params = dict(CATBOOST_PARAMS)
final_params["iterations"] = best_iterations
final_params["verbose"] = 200
final_model = CatBoostRegressor(**final_params)
final_model.fit(Pool(X_full, label=final_target, cat_features=full_cat_columns))

test_prediction = final_model.predict(X_test)
if final_trend_model is not None:
    test_prediction = test_prediction + final_trend_model.predict(ordered_test[["MONTH_INDEX"]])
test_prediction = np.clip(test_prediction, a_min=0, a_max=None)

submission = pd.DataFrame({
    "Id": ordered_test["ROW_ID"].astype(int),
    "Predicted": test_prediction,
}).sort_values("Id").reset_index(drop=True)

assert len(submission) == 50_000
assert submission["Id"].equals(pd.Series(np.arange(50_000), name="Id"))
assert submission["Predicted"].notna().all()

model_path = MODEL_ROOT / "catboost_team_main.cbm"
submission_path = MODEL_ROOT / "submission_catboost.csv"
metrics_path = MODEL_ROOT / "validation_metrics.csv"
importance_path = MODEL_ROOT / "feature_importance.csv"
metadata_path = MODEL_ROOT / "model_metadata.json"

final_model.save_model(model_path)
submission.to_csv(submission_path, index=False)
results_df.to_csv(metrics_path, index=False)
feature_importance.to_csv(importance_path, index=False)

metadata = {
    "selected_experiment": BEST_SAFE_EXPERIMENT,
    "target_mode": best_target_mode,
    "macro_used_in_final_model": False,
    "macro_reason": "Realized test-period macro data are unavailable at one fixed forecast origin.",
    "iterations": best_iterations,
    "validation_start": str(validation_start.date()),
    "validation_metrics": {
        key: float(best_row[key]) for key in ["MAE", "RMSE", "R2", "BIAS"]
    },
    "trend_intercept": float(final_trend_model.intercept_) if final_trend_model is not None else None,
    "trend_monthly_slope": float(final_trend_model.coef_[0]) if final_trend_model is not None else None,
    "feature_count": int(X_full.shape[1]),
    "categorical_feature_count": int(len(full_cat_columns)),
    "train_rows": int(len(full_train)),
    "test_rows": int(len(ordered_test)),
    "random_seed": RANDOM_SEED,
    "catboost_version": catboost.__version__,
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(json.dumps(metadata, indent=2))
display(submission.head())
print(f"Saved model:      {model_path}")
print(f"Saved submission: {submission_path}")

## 10. Improvement roadmap

The following changes should be evaluated one at a time with the same forward
validation design:

1. **Multiple rolling folds:** confirm that target design and macro gains repeat
   across at least two earlier forecast windows.
2. **Hyperparameter search:** tune depth, learning rate, L2 regularization,
   randomness, and categorical-combination complexity with time-based folds.
3. **Leakage-safe prior-rent features:** add address, town-flat-type, and
   flat-type histories calculated from strictly earlier months.
4. **Forecast-compatible macro inputs:** replace realized test-period macro
   values with values frozen or forecast at the declared origin.
5. **Trend modeling:** compare the linear monthly trend with robust regression,
   splines constrained to extrapolate sensibly, or a separately forecast market
   index.
6. **Group ablation:** remove whole correlated groups—demographics, traffic,
   schools, malls, clusters—rather than trusting individual importance ranks.
7. **Segment diagnostics:** report error for unseen addresses, towns, flat
   types, and price bands. High-cardinality categories can otherwise hide poor
   generalization.
8. **Ensembling:** average CatBoost with a linear/GAM trend model or a LightGBM
   model only when forward-fold errors are sufficiently different.

The most important limitation remains time extrapolation. Increasing tree depth
or iteration count cannot solve it by itself; the validation framework and
market-level trend component matter more.

## 11. Advanced time-aware model search

This section implements the next set of experiments without changing or overwriting
the submitted CatBoost-raw model above. It adds:

1. log-rent modelling, including a smearing correction after exponentiation;
2. exponentially decaying weights so older rows remain available but receive less
   influence;
3. a composition-adjusted monthly market index based on fixed town-flat-type
   reference weights;
4. last-level, damped Holt, seasonal damped Holt, local-linear state-space, and
   robust recent-drift trend forecasts;
5. partially pooled town-flat-type trend adjustments;
6. expanding-window validation with several future-like cutoffs;
7. an out-of-fold optimized blend of the strongest raw/log model and strongest
   trend-residual model; and
8. separate advanced submission files and diagnostics.

Leakage rule: every validation trend, segment adjustment, smearing factor, weight,
and CatBoost model is estimated using data strictly before that fold's validation
start. Realized macroeconomic values are excluded because their future test-period
values would not have been known at the fixed submission origin.

The default quick mode confirms that the pipeline runs without fitting every
combination. Set ADVANCED_FULL_SEARCH to True for the full model-selection run.
The full search can take considerably longer because each candidate is refitted in
every rolling fold.

In [ ]:
from sklearn.linear_model import HuberRegressor

try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    from statsmodels.tsa.statespace.structural import UnobservedComponents
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This section requires statsmodels. Install it in the active notebook "
        "environment with: %pip install statsmodels"
    ) from exc

ADVANCED_FULL_SEARCH = False
ADVANCED_HORIZON_MONTHS = 12
ADVANCED_FOLD_STEP_MONTHS = 6
ADVANCED_N_FOLDS = 3 if ADVANCED_FULL_SEARCH else 2
ADVANCED_ITERATIONS = 600 if ADVANCED_FULL_SEARCH else 300
ADVANCED_EARLY_STOPPING = 75
ADVANCED_SEGMENT_COLUMNS = ["TOWN", "FLAT_TYPE"]
ADVANCED_SEGMENT_PRIOR_MONTHS = 12
ADVANCED_SEGMENT_SLOPE_LIMIT = 0.010
ADVANCED_TREND_DAMPING = 0.92

if ADVANCED_FULL_SEARCH:
    ADVANCED_RECENCY_HALF_LIVES = [None, 12, 24, 36]
    ADVANCED_LOG_HALF_LIVES = [None, 24]
    ADVANCED_TREND_METHODS = [
        "last_level",
        "damped_holt",
        "damped_holt_seasonal",
        "local_linear",
        "recent_robust",
    ]
    ADVANCED_HIERARCHICAL_OPTIONS = [False, True]
else:
    ADVANCED_RECENCY_HALF_LIVES = [None, 24]
    ADVANCED_LOG_HALF_LIVES = [None]
    ADVANCED_TREND_METHODS = ["last_level", "damped_holt", "local_linear"]
    ADVANCED_HIERARCHICAL_OPTIONS = [True]


def make_expanding_month_folds(
    frame,
    horizon_months=12,
    n_folds=3,
    step_months=6,
    minimum_training_months=24,
):
    months = pd.DatetimeIndex(sorted(frame["RENT_APPROVAL_DATE"].unique()))
    final_start = len(months) - horizon_months
    starts = [
        final_start - step_months * offset
        for offset in reversed(range(n_folds))
    ]
    if min(starts) < minimum_training_months:
        raise ValueError(
            "Insufficient history for the requested folds. Reduce n_folds or "
            "horizon_months."
        )

    folds = []
    for fold_number, start in enumerate(starts, start=1):
        valid_months = months[start:start + horizon_months]
        fit_part = frame.loc[
            frame["RENT_APPROVAL_DATE"] < valid_months[0]
        ].sort_values(["RENT_APPROVAL_DATE", "ROW_ID"]).reset_index(drop=True)
        valid_part = frame.loc[
            frame["RENT_APPROVAL_DATE"].isin(valid_months)
        ].sort_values(["RENT_APPROVAL_DATE", "ROW_ID"]).reset_index(drop=True)
        assert fit_part["RENT_APPROVAL_DATE"].max() < valid_part["RENT_APPROVAL_DATE"].min()
        folds.append({
            "fold": fold_number,
            "fit": fit_part,
            "valid": valid_part,
            "fit_end": fit_part["RENT_APPROVAL_DATE"].max(),
            "valid_start": valid_part["RENT_APPROVAL_DATE"].min(),
            "valid_end": valid_part["RENT_APPROVAL_DATE"].max(),
        })
    return folds


advanced_folds = make_expanding_month_folds(
    train,
    horizon_months=ADVANCED_HORIZON_MONTHS,
    n_folds=ADVANCED_N_FOLDS,
    step_months=ADVANCED_FOLD_STEP_MONTHS,
)

display(pd.DataFrame([
    {
        "fold": fold["fold"],
        "fit_rows": len(fold["fit"]),
        "fit_end": fold["fit_end"],
        "validation_rows": len(fold["valid"]),
        "validation_start": fold["valid_start"],
        "validation_end": fold["valid_end"],
    }
    for fold in advanced_folds
]))

### 11.1 Composition-adjusted market trend

A raw monthly mean or median changes when the mix of towns and flat types changes.
The index below first calculates monthly median log rent within each town-flat-type
segment. It then combines those segment medians using fixed weights calculated
from the fitting window. This holds the reference composition approximately
constant over time.

The trend methods forecast this common index. The optional hierarchical adjustment
estimates each segment's deviation and recent slope, then shrinks both toward the
global trend. Sparse segments therefore cannot create extreme forecasts.

In [ ]:
def segment_key(frame, columns=ADVANCED_SEGMENT_COLUMNS):
    values = frame[columns[0]].fillna("__MISSING__").astype(str)
    for column in columns[1:]:
        values = values.str.cat(
            frame[column].fillna("__MISSING__").astype(str),
            sep=" | ",
        )
    return values


def build_composition_adjusted_log_index(
    frame,
    segment_columns=ADVANCED_SEGMENT_COLUMNS,
):
    work = frame[["RENT_APPROVAL_DATE", TARGET, *segment_columns]].copy()
    work["_SEGMENT"] = segment_key(work, segment_columns)
    work["_LOG_RENT"] = np.log(work[TARGET].clip(lower=1.0))

    reference_weights = (
        work["_SEGMENT"].value_counts(normalize=True).rename("_REFERENCE_WEIGHT")
    )
    monthly_segment = (
        work.groupby(["RENT_APPROVAL_DATE", "_SEGMENT"], as_index=False)["_LOG_RENT"]
        .median()
        .merge(
            reference_weights.rename_axis("_SEGMENT").reset_index(),
            on="_SEGMENT",
            how="left",
            validate="many_to_one",
        )
    )
    monthly_segment["_WEIGHTED_LOG_RENT"] = (
        monthly_segment["_LOG_RENT"] * monthly_segment["_REFERENCE_WEIGHT"]
    )
    numerator = monthly_segment.groupby("RENT_APPROVAL_DATE")[
        "_WEIGHTED_LOG_RENT"
    ].sum()
    denominator = monthly_segment.groupby("RENT_APPROVAL_DATE")[
        "_REFERENCE_WEIGHT"
    ].sum()
    index = (numerator / denominator).sort_index()
    complete_months = pd.date_range(index.index.min(), index.index.max(), freq="MS")
    index = index.reindex(complete_months).interpolate(limit_direction="both")
    index.name = "LOG_MARKET_INDEX"
    return index, reference_weights


def fit_log_trend_forecaster(monthly_index, future_months, method):
    y = monthly_index.astype(float).asfreq("MS").interpolate(limit_direction="both")
    future_months = pd.DatetimeIndex(sorted(pd.to_datetime(future_months)))
    horizon = len(future_months)
    details = {"requested_method": method, "method_used": method}

    if method == "last_level":
        fitted = y.copy()
        level = float(y.tail(min(3, len(y))).mean())
        forecast = pd.Series(level, index=future_months, dtype=float)

    elif method in {"damped_holt", "damped_holt_seasonal"}:
        use_seasonality = method.endswith("seasonal") and len(y) >= 36
        if method.endswith("seasonal") and not use_seasonality:
            details["method_used"] = "damped_holt"
        fitted_model = ExponentialSmoothing(
            y,
            trend="add",
            damped_trend=True,
            seasonal="add" if use_seasonality else None,
            seasonal_periods=12 if use_seasonality else None,
            initialization_method="estimated",
        ).fit(optimized=True, use_brute=True)
        fitted = pd.Series(fitted_model.fittedvalues, index=y.index, dtype=float)
        forecast = pd.Series(
            np.asarray(fitted_model.forecast(horizon), dtype=float),
            index=future_months,
        )

    elif method == "local_linear":
        try:
            fitted_model = UnobservedComponents(
                y,
                level="local linear trend",
                seasonal=12 if len(y) >= 36 else None,
            ).fit(disp=False)
            fitted = pd.Series(fitted_model.fittedvalues, index=y.index, dtype=float)
            forecast = pd.Series(
                np.asarray(
                    fitted_model.get_forecast(steps=horizon).predicted_mean,
                    dtype=float,
                ),
                index=future_months,
            )
        except Exception as exc:
            warnings.warn(
                f"Local-linear trend failed ({exc}); using damped Holt instead."
            )
            return fit_log_trend_forecaster(y, future_months, "damped_holt")

    elif method == "recent_robust":
        x = np.arange(len(y), dtype=float)
        window = min(24, len(y))
        robust_model = HuberRegressor(epsilon=1.35, alpha=0.01)
        robust_model.fit(x[-window:, None], y.to_numpy()[-window:])
        slope = float(
            np.clip(
                robust_model.coef_[0],
                -ADVANCED_SEGMENT_SLOPE_LIMIT,
                ADVANCED_SEGMENT_SLOPE_LIMIT,
            )
        )
        cutoff_level = float(robust_model.predict([[x[-1]]])[0])
        fitted = pd.Series(robust_model.predict(x[:, None]), index=y.index)
        steps = np.arange(1, horizon + 1, dtype=float)
        damped_steps = (
            (1 - ADVANCED_TREND_DAMPING ** steps)
            / (1 - ADVANCED_TREND_DAMPING)
        )
        forecast = pd.Series(
            cutoff_level + slope * damped_steps,
            index=future_months,
        )
        details["recent_monthly_slope"] = slope

    else:
        raise ValueError(f"Unknown trend method: {method}")

    fitted = fitted.reindex(y.index).replace([np.inf, -np.inf], np.nan)
    fitted = fitted.interpolate(limit_direction="both").fillna(y)
    if horizon:
        forecast = forecast.replace([np.inf, -np.inf], np.nan)
        forecast = forecast.interpolate(limit_direction="both")
        if forecast.isna().any():
            raise ValueError(f"Trend method {method} returned missing forecasts.")
    return fitted, forecast, details


def fit_segment_trend_adjustments(
    frame,
    global_history,
    segment_columns=ADVANCED_SEGMENT_COLUMNS,
    prior_months=ADVANCED_SEGMENT_PRIOR_MONTHS,
):
    work = frame[["RENT_APPROVAL_DATE", TARGET, *segment_columns]].copy()
    work["_SEGMENT"] = segment_key(work, segment_columns)
    work["_LOG_RENT"] = np.log(work[TARGET].clip(lower=1.0))
    work["_GLOBAL"] = work["RENT_APPROVAL_DATE"].map(global_history)
    work["_DEVIATION"] = work["_LOG_RENT"] - work["_GLOBAL"]
    cutoff = pd.Timestamp(global_history.index.max())

    monthly = (
        work.groupby(["_SEGMENT", "RENT_APPROVAL_DATE"], as_index=False)["_DEVIATION"]
        .median()
    )
    parameters = {}
    for group_name, group in monthly.groupby("_SEGMENT"):
        group = group.sort_values("RENT_APPROVAL_DATE")
        month_distance = (
            (group["RENT_APPROVAL_DATE"].dt.year - cutoff.year) * 12
            + group["RENT_APPROVAL_DATE"].dt.month
            - cutoff.month
        ).to_numpy(dtype=float)
        values = group["_DEVIATION"].to_numpy(dtype=float)
        n_months = len(group)

        if n_months >= 6 and np.ptp(month_distance) > 0:
            recent = min(24, n_months)
            recent_x = month_distance[-recent:]
            recent_y = values[-recent:]
            median = float(np.median(recent_y))
            mad = float(np.median(np.abs(recent_y - median)))
            if mad > 0:
                recent_y = np.clip(
                    recent_y,
                    median - 3 * 1.4826 * mad,
                    median + 3 * 1.4826 * mad,
                )
            slope, offset = np.polyfit(recent_x, recent_y, deg=1)
            offset = float(offset)
            slope = float(slope)
        else:
            offset = float(np.median(values))
            slope = 0.0

        shrinkage = n_months / (n_months + prior_months)
        parameters[group_name] = {
            "offset": float(np.clip(offset, -0.35, 0.35)),
            "slope": float(
                np.clip(
                    slope,
                    -ADVANCED_SEGMENT_SLOPE_LIMIT,
                    ADVANCED_SEGMENT_SLOPE_LIMIT,
                )
            ),
            "shrinkage": float(shrinkage),
            "months": int(n_months),
        }
    return parameters, cutoff


def apply_segment_trend_adjustments(
    frame,
    parameters,
    cutoff,
    segment_columns=ADVANCED_SEGMENT_COLUMNS,
):
    keys = segment_key(frame, segment_columns)
    month_distance = (
        (frame["RENT_APPROVAL_DATE"].dt.year - cutoff.year) * 12
        + frame["RENT_APPROVAL_DATE"].dt.month
        - cutoff.month
    ).to_numpy(dtype=float)

    parameter_frame = pd.DataFrame.from_dict(parameters, orient="index")
    offsets = keys.map(parameter_frame["offset"]).fillna(0).to_numpy(float)
    slopes = keys.map(parameter_frame["slope"]).fillna(0).to_numpy(float)
    shrinkage = (
        keys.map(parameter_frame["shrinkage"]).fillna(0).to_numpy(float)
    )
    positive_distance = np.maximum(month_distance, 0)
    damped_distance = np.where(
        month_distance <= 0,
        month_distance,
        (1 - ADVANCED_TREND_DAMPING ** positive_distance)
        / (1 - ADVANCED_TREND_DAMPING),
    )
    return shrinkage * (offsets + slopes * damped_distance)


def make_row_log_trends(
    fit_frame,
    score_frame,
    method,
    hierarchical,
):
    monthly_index, reference_weights = build_composition_adjusted_log_index(fit_frame)
    last_observed_month = pd.Timestamp(monthly_index.index.max())
    score_months = pd.DatetimeIndex(
        sorted(score_frame["RENT_APPROVAL_DATE"].unique())
    )
    future_months = score_months[score_months > last_observed_month]
    history, future, details = fit_log_trend_forecaster(
        monthly_index,
        future_months,
        method,
    )

    score_global_map = history.copy()
    score_global_map = pd.concat([score_global_map, future])
    fit_log_trend = (
        fit_frame["RENT_APPROVAL_DATE"].map(history).to_numpy(dtype=float)
    )
    score_log_trend = (
        score_frame["RENT_APPROVAL_DATE"].map(score_global_map).to_numpy(dtype=float)
    )

    segment_parameters = {}
    segment_cutoff = last_observed_month
    if hierarchical:
        segment_parameters, segment_cutoff = fit_segment_trend_adjustments(
            fit_frame,
            history,
        )
        fit_log_trend = fit_log_trend + apply_segment_trend_adjustments(
            fit_frame,
            segment_parameters,
            segment_cutoff,
        )
        score_log_trend = score_log_trend + apply_segment_trend_adjustments(
            score_frame,
            segment_parameters,
            segment_cutoff,
        )

    if not np.isfinite(fit_log_trend).all() or not np.isfinite(score_log_trend).all():
        raise ValueError("Trend construction produced non-finite row-level values.")

    bundle = {
        "monthly_index": monthly_index,
        "history": history,
        "future": future,
        "reference_weights": reference_weights,
        "segment_parameters": segment_parameters,
        "segment_cutoff": segment_cutoff,
        "details": details,
    }
    return fit_log_trend, score_log_trend, bundle

### 11.2 Candidate models and leakage-safe fitting

Raw candidates keep the original feature design. Log-raw candidates model log rent
directly. Trend-residual candidates remove absolute year and month-index predictors
from CatBoost so the separately forecast trend owns long-run extrapolation; month
of year remains available for seasonality.

Recency weights use an exponential half-life. A 24-month half-life means a row 24
months older than the fitting cutoff receives half the weight of a current row.
No historical rows are discarded.

In [ ]:
ADVANCED_ABSOLUTE_TIME_COLUMNS = {
    "MONTH_INDEX",
    "RENT_APPROVAL_YEAR",
}


def make_residual_feature_frame(frame):
    X, categorical_columns = make_feature_frame(frame, include_macro=False)
    drop_columns = [
        column for column in ADVANCED_ABSOLUTE_TIME_COLUMNS if column in X.columns
    ]
    X = X.drop(columns=drop_columns)
    categorical_columns = [
        column for column in categorical_columns if column in X.columns
    ]
    return X, categorical_columns


def make_recency_weights(frame, half_life_months):
    if half_life_months is None:
        return None
    age = (
        frame["MONTH_INDEX"].max() - frame["MONTH_INDEX"]
    ).to_numpy(dtype=float)
    weights = 0.5 ** (age / float(half_life_months))
    return weights / weights.mean()


def advanced_catboost_parameters(iterations=ADVANCED_ITERATIONS, verbose=False):
    return {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "iterations": int(iterations),
        "learning_rate": 0.05,
        "depth": 8,
        "l2_leaf_reg": 8.0,
        "random_strength": 0.5,
        "bagging_temperature": 0.5,
        "random_seed": RANDOM_SEED,
        "has_time": True,
        "allow_writing_files": False,
        "thread_count": -1,
        "verbose": 100 if verbose else False,
    }


def log_smearing_factor(actual_log, fitted_log):
    residual = np.asarray(actual_log) - np.asarray(fitted_log)
    residual = np.clip(residual, -0.75, 0.75)
    return float(np.mean(np.exp(residual)))


def build_advanced_candidate_specs():
    specifications = []
    for half_life in ADVANCED_RECENCY_HALF_LIVES:
        suffix = "unweighted" if half_life is None else f"half_life_{half_life}m"
        specifications.append({
            "name": f"raw_{suffix}",
            "family": "raw",
            "kind": "raw",
            "half_life": half_life,
        })
    for half_life in ADVANCED_LOG_HALF_LIVES:
        suffix = "unweighted" if half_life is None else f"half_life_{half_life}m"
        specifications.append({
            "name": f"log_raw_{suffix}",
            "family": "log_raw",
            "kind": "log_raw",
            "half_life": half_life,
        })
    for method in ADVANCED_TREND_METHODS:
        for hierarchical in ADVANCED_HIERARCHICAL_OPTIONS:
            level = "hierarchical" if hierarchical else "global"
            specifications.append({
                "name": f"trend_{method}_{level}",
                "family": "trend_residual",
                "kind": "trend_residual",
                "method": method,
                "hierarchical": hierarchical,
                "half_life": 24,
            })
    return specifications


ADVANCED_CANDIDATE_SPECS = build_advanced_candidate_specs()
ADVANCED_SPEC_BY_NAME = {
    specification["name"]: specification
    for specification in ADVANCED_CANDIDATE_SPECS
}
display(pd.DataFrame(ADVANCED_CANDIDATE_SPECS))


def fit_predict_advanced_fold(specification, fit_frame, valid_frame):
    kind = specification["kind"]
    if kind == "trend_residual":
        X_fit, categorical_columns = make_residual_feature_frame(fit_frame)
        X_valid, valid_categorical_columns = make_residual_feature_frame(valid_frame)
    else:
        X_fit, categorical_columns = make_feature_frame(
            fit_frame,
            include_macro=False,
        )
        X_valid, valid_categorical_columns = make_feature_frame(
            valid_frame,
            include_macro=False,
        )
    assert list(X_fit.columns) == list(X_valid.columns)
    assert categorical_columns == valid_categorical_columns

    fit_weights = make_recency_weights(
        fit_frame,
        specification.get("half_life"),
    )
    trend_bundle = None

    if kind == "raw":
        y_fit = fit_frame[TARGET].to_numpy(dtype=float)
        y_valid = valid_frame[TARGET].to_numpy(dtype=float)
        validation_offset = None
    elif kind == "log_raw":
        y_fit = np.log(fit_frame[TARGET].clip(lower=1.0)).to_numpy(dtype=float)
        y_valid = np.log(valid_frame[TARGET].clip(lower=1.0)).to_numpy(dtype=float)
        validation_offset = None
    elif kind == "trend_residual":
        fit_offset, validation_offset, trend_bundle = make_row_log_trends(
            fit_frame,
            valid_frame,
            method=specification["method"],
            hierarchical=specification["hierarchical"],
        )
        y_fit = (
            np.log(fit_frame[TARGET].clip(lower=1.0)).to_numpy(dtype=float)
            - fit_offset
        )
        y_valid = (
            np.log(valid_frame[TARGET].clip(lower=1.0)).to_numpy(dtype=float)
            - validation_offset
        )
    else:
        raise ValueError(kind)

    fit_pool = Pool(
        X_fit,
        label=y_fit,
        cat_features=categorical_columns,
        weight=fit_weights,
    )
    valid_pool = Pool(
        X_valid,
        label=y_valid,
        cat_features=categorical_columns,
    )
    model = CatBoostRegressor(**advanced_catboost_parameters())
    model.fit(
        fit_pool,
        eval_set=valid_pool,
        use_best_model=True,
        early_stopping_rounds=ADVANCED_EARLY_STOPPING,
    )

    raw_prediction = model.predict(X_valid)
    smearing = 1.0
    if kind == "raw":
        prediction = raw_prediction
    else:
        fitted_log = model.predict(X_fit)
        if kind == "trend_residual":
            fitted_log = fitted_log + fit_offset
            prediction_log = raw_prediction + validation_offset
            actual_fit_log = np.log(
                fit_frame[TARGET].clip(lower=1.0)
            ).to_numpy(dtype=float)
        else:
            prediction_log = raw_prediction
            actual_fit_log = y_fit
        smearing = log_smearing_factor(actual_fit_log, fitted_log)
        prediction = np.exp(prediction_log) * smearing

    prediction = np.clip(np.asarray(prediction, dtype=float), 0, None)
    return {
        "prediction": prediction,
        "best_iteration": max(int(model.get_best_iteration() + 1), 50),
        "smearing": smearing,
        "trend_bundle": trend_bundle,
    }

In [ ]:
advanced_fold_rows = []
advanced_prediction_frames = []

for fold in advanced_folds:
    print(
        f"Fold {fold['fold']}: fit through {fold['fit_end'].date()}, "
        f"validate {fold['valid_start'].date()} to {fold['valid_end'].date()}"
    )
    for specification in ADVANCED_CANDIDATE_SPECS:
        outcome = fit_predict_advanced_fold(
            specification,
            fold["fit"],
            fold["valid"],
        )
        metrics = regression_metrics(
            fold["valid"][TARGET],
            outcome["prediction"],
        )
        advanced_fold_rows.append({
            "fold": fold["fold"],
            "fit_end": fold["fit_end"],
            "validation_start": fold["valid_start"],
            "validation_end": fold["valid_end"],
            "candidate": specification["name"],
            "family": specification["family"],
            "best_iteration": outcome["best_iteration"],
            "smearing": outcome["smearing"],
            **metrics,
        })
        prediction_frame = fold["valid"][
            ["ROW_ID", "RENT_APPROVAL_DATE", TARGET]
        ].copy()
        prediction_frame["fold"] = fold["fold"]
        prediction_frame["candidate"] = specification["name"]
        prediction_frame["PREDICTION"] = outcome["prediction"]
        advanced_prediction_frames.append(prediction_frame)
        print(
            f"  {specification['name']}: "
            f"RMSE={metrics['RMSE']:,.2f}, bias={metrics['BIAS']:,.2f}"
        )

advanced_cv_results = pd.DataFrame(advanced_fold_rows)
advanced_oof_predictions = pd.concat(
    advanced_prediction_frames,
    ignore_index=True,
)

advanced_summary = (
    advanced_cv_results.groupby(["candidate", "family"], as_index=False)
    .agg(
        mean_RMSE=("RMSE", "mean"),
        std_RMSE=("RMSE", "std"),
        worst_RMSE=("RMSE", "max"),
        mean_MAE=("MAE", "mean"),
        mean_R2=("R2", "mean"),
        mean_BIAS=("BIAS", "mean"),
        median_best_iteration=("best_iteration", "median"),
    )
    .sort_values(["mean_RMSE", "worst_RMSE"])
    .reset_index(drop=True)
)

raw_families = {"raw", "log_raw"}
best_raw_candidate = (
    advanced_summary.loc[advanced_summary["family"].isin(raw_families)]
    .iloc[0]["candidate"]
)
best_trend_candidate = (
    advanced_summary.loc[advanced_summary["family"].eq("trend_residual")]
    .iloc[0]["candidate"]
)

raw_oof = advanced_oof_predictions.loc[
    advanced_oof_predictions["candidate"].eq(best_raw_candidate),
    ["fold", "ROW_ID", "RENT_APPROVAL_DATE", TARGET, "PREDICTION"],
].rename(columns={"PREDICTION": "RAW_PREDICTION"})
trend_oof = advanced_oof_predictions.loc[
    advanced_oof_predictions["candidate"].eq(best_trend_candidate),
    ["fold", "ROW_ID", "PREDICTION"],
].rename(columns={"PREDICTION": "TREND_PREDICTION"})
blend_oof = raw_oof.merge(
    trend_oof,
    on=["fold", "ROW_ID"],
    how="inner",
    validate="one_to_one",
)

blend_grid = np.linspace(0.0, 1.0, 101)
blend_grid_rmse = []
for raw_weight in blend_grid:
    candidate_prediction = (
        raw_weight * blend_oof["RAW_PREDICTION"]
        + (1 - raw_weight) * blend_oof["TREND_PREDICTION"]
    )
    blend_grid_rmse.append(
        mean_squared_error(blend_oof[TARGET], candidate_prediction) ** 0.5
    )
ADVANCED_RAW_BLEND_WEIGHT = float(blend_grid[int(np.argmin(blend_grid_rmse))])
blend_oof["PREDICTION"] = (
    ADVANCED_RAW_BLEND_WEIGHT * blend_oof["RAW_PREDICTION"]
    + (1 - ADVANCED_RAW_BLEND_WEIGHT) * blend_oof["TREND_PREDICTION"]
)

blend_fold_rows = []
for fold_number, group in blend_oof.groupby("fold"):
    metrics = regression_metrics(group[TARGET], group["PREDICTION"])
    source_fold = advanced_folds[int(fold_number) - 1]
    blend_fold_rows.append({
        "fold": int(fold_number),
        "fit_end": source_fold["fit_end"],
        "validation_start": source_fold["valid_start"],
        "validation_end": source_fold["valid_end"],
        "candidate": "optimized_raw_trend_blend",
        "family": "blend",
        "best_iteration": np.nan,
        "smearing": np.nan,
        **metrics,
    })

advanced_cv_results = pd.concat(
    [advanced_cv_results, pd.DataFrame(blend_fold_rows)],
    ignore_index=True,
)
advanced_summary = (
    advanced_cv_results.groupby(["candidate", "family"], as_index=False)
    .agg(
        mean_RMSE=("RMSE", "mean"),
        std_RMSE=("RMSE", "std"),
        worst_RMSE=("RMSE", "max"),
        mean_MAE=("MAE", "mean"),
        mean_R2=("R2", "mean"),
        mean_BIAS=("BIAS", "mean"),
        median_best_iteration=("best_iteration", "median"),
    )
    .sort_values(["mean_RMSE", "worst_RMSE"])
    .reset_index(drop=True)
)
ADVANCED_SELECTED_MODEL = advanced_summary.iloc[0]["candidate"]

print(f"Best raw/log candidate:   {best_raw_candidate}")
print(f"Best trend candidate:     {best_trend_candidate}")
print(f"Optimal raw blend weight: {ADVANCED_RAW_BLEND_WEIGHT:.2f}")
print(f"Selected advanced model:  {ADVANCED_SELECTED_MODEL}")
display(advanced_summary)

### 11.3 Rolling-validation diagnostics

Mean RMSE is the primary ranking statistic. Worst-fold RMSE and fold-to-fold
variation are retained because a model that wins one market regime but fails badly
in another is risky for the unseen Kaggle period. The blend weight is estimated
only from rolling out-of-fold predictions.

The second chart shows the latest validation window because it is closest to the
official test origin. It should still be interpreted together with all earlier
folds rather than used as the sole selection criterion.

In [ ]:
plot_summary = advanced_summary.sort_values("mean_RMSE")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(
    plot_summary["candidate"],
    plot_summary["mean_RMSE"],
    xerr=plot_summary["std_RMSE"].fillna(0),
    color=np.where(
        plot_summary["family"].eq("trend_residual"),
        "#59A14F",
        np.where(plot_summary["family"].eq("blend"), "#F28E2B", "#4C78A8"),
    ),
    alpha=0.9,
)
axes[0].invert_yaxis()
axes[0].set_xlabel("Mean rolling RMSE (SGD)")
axes[0].set_title("Advanced model comparison")
axes[0].grid(axis="x", alpha=0.25)

latest_fold_number = max(fold["fold"] for fold in advanced_folds)
latest_actual = advanced_folds[-1]["valid"][
    ["ROW_ID", "RENT_APPROVAL_DATE", TARGET]
].copy()
latest_candidates = {
    best_raw_candidate: advanced_oof_predictions.loc[
        (advanced_oof_predictions["fold"].eq(latest_fold_number))
        & (advanced_oof_predictions["candidate"].eq(best_raw_candidate)),
        ["ROW_ID", "PREDICTION"],
    ],
    best_trend_candidate: advanced_oof_predictions.loc[
        (advanced_oof_predictions["fold"].eq(latest_fold_number))
        & (advanced_oof_predictions["candidate"].eq(best_trend_candidate)),
        ["ROW_ID", "PREDICTION"],
    ],
    "optimized_raw_trend_blend": blend_oof.loc[
        blend_oof["fold"].eq(latest_fold_number),
        ["ROW_ID", "PREDICTION"],
    ],
}

monthly_plot = (
    latest_actual.groupby("RENT_APPROVAL_DATE", as_index=False)[TARGET]
    .mean()
    .rename(columns={TARGET: "Actual"})
)
axes[1].plot(
    monthly_plot["RENT_APPROVAL_DATE"],
    monthly_plot["Actual"],
    color="black",
    linewidth=3,
    marker="o",
    label="Actual",
)
for candidate_name, prediction_table in latest_candidates.items():
    joined = latest_actual.merge(
        prediction_table,
        on="ROW_ID",
        how="left",
        validate="one_to_one",
    )
    monthly_candidate = (
        joined.groupby("RENT_APPROVAL_DATE", as_index=False)["PREDICTION"].mean()
    )
    axes[1].plot(
        monthly_candidate["RENT_APPROVAL_DATE"],
        monthly_candidate["PREDICTION"],
        marker=".",
        linewidth=1.8,
        label=candidate_name,
    )
axes[1].set_title("Latest rolling fold: monthly actual vs predicted")
axes[1].set_ylabel("Mean monthly rent (SGD)")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

best_trend_specification = ADVANCED_SPEC_BY_NAME[best_trend_candidate]
latest_fit = advanced_folds[-1]["fit"]
latest_valid = advanced_folds[-1]["valid"]
_, _, latest_trend_bundle = make_row_log_trends(
    latest_fit,
    latest_valid,
    method=best_trend_specification["method"],
    hierarchical=best_trend_specification["hierarchical"],
)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(
    latest_trend_bundle["monthly_index"].index,
    np.exp(latest_trend_bundle["monthly_index"]),
    color="black",
    marker="o",
    label="Composition-adjusted observed index",
)
ax.plot(
    latest_trend_bundle["history"].index,
    np.exp(latest_trend_bundle["history"]),
    color="#4C78A8",
    linewidth=2,
    label="Fitted common trend",
)
ax.plot(
    latest_trend_bundle["future"].index,
    np.exp(latest_trend_bundle["future"]),
    color="#E15759",
    linewidth=2,
    marker="o",
    label="Forecast common trend",
)
ax.axvline(latest_fit["RENT_APPROVAL_DATE"].max(), color="grey", linestyle="--")
ax.set_title(
    f"Best trend method in latest fold: {best_trend_candidate}"
)
ax.set_ylabel("Composition-adjusted rent level (SGD)")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### 11.4 Refit finalists and create advanced submissions

The strongest raw/log candidate and strongest trend-residual candidate are refitted
on all labelled rows. Their iteration counts are the medians selected in the rolling
folds. The optimized out-of-fold blend weight is then applied unchanged to the
official test predictions.

This cell writes separate files and does not overwrite submission_catboost.csv.
The selected advanced submission should only replace the existing Kaggle choice
when its rolling validation is convincingly better and reasonably stable across
folds.

In [ ]:
def candidate_iterations(candidate_name):
    rows = advanced_cv_results.loc[
        advanced_cv_results["candidate"].eq(candidate_name),
        "best_iteration",
    ].dropna()
    if rows.empty:
        return ADVANCED_ITERATIONS
    return max(int(np.median(rows)), 50)


def fit_final_advanced_candidate(
    specification,
    fit_frame,
    score_frame,
    iterations,
):
    kind = specification["kind"]
    if kind == "trend_residual":
        X_fit, categorical_columns = make_residual_feature_frame(fit_frame)
        X_score, score_categorical_columns = make_residual_feature_frame(score_frame)
    else:
        X_fit, categorical_columns = make_feature_frame(
            fit_frame,
            include_macro=False,
        )
        X_score, score_categorical_columns = make_feature_frame(
            score_frame,
            include_macro=False,
        )
    assert list(X_fit.columns) == list(X_score.columns)
    assert categorical_columns == score_categorical_columns

    weights = make_recency_weights(
        fit_frame,
        specification.get("half_life"),
    )
    trend_bundle = None

    if kind == "raw":
        y_fit = fit_frame[TARGET].to_numpy(dtype=float)
        fit_offset = None
        score_offset = None
    elif kind == "log_raw":
        y_fit = np.log(fit_frame[TARGET].clip(lower=1.0)).to_numpy(dtype=float)
        fit_offset = None
        score_offset = None
    elif kind == "trend_residual":
        fit_offset, score_offset, trend_bundle = make_row_log_trends(
            fit_frame,
            score_frame,
            method=specification["method"],
            hierarchical=specification["hierarchical"],
        )
        y_fit = (
            np.log(fit_frame[TARGET].clip(lower=1.0)).to_numpy(dtype=float)
            - fit_offset
        )
    else:
        raise ValueError(kind)

    model = CatBoostRegressor(
        **advanced_catboost_parameters(iterations=iterations, verbose=True)
    )
    model.fit(
        Pool(
            X_fit,
            label=y_fit,
            cat_features=categorical_columns,
            weight=weights,
        )
    )
    score_raw = model.predict(X_score)
    smearing = 1.0

    if kind == "raw":
        prediction = score_raw
    else:
        fitted_log = model.predict(X_fit)
        if kind == "trend_residual":
            fitted_log = fitted_log + fit_offset
            prediction_log = score_raw + score_offset
        else:
            prediction_log = score_raw
        actual_fit_log = np.log(
            fit_frame[TARGET].clip(lower=1.0)
        ).to_numpy(dtype=float)
        smearing = log_smearing_factor(actual_fit_log, fitted_log)
        prediction = np.exp(prediction_log) * smearing

    prediction = np.clip(np.asarray(prediction, dtype=float), 0, None)
    return {
        "model": model,
        "prediction": prediction,
        "smearing": smearing,
        "trend_bundle": trend_bundle,
        "iterations": int(iterations),
    }


advanced_full_train = train.sort_values(
    ["RENT_APPROVAL_DATE", "ROW_ID"]
).reset_index(drop=True)
advanced_ordered_test = test.sort_values("ROW_ID").reset_index(drop=True)

best_raw_specification = ADVANCED_SPEC_BY_NAME[best_raw_candidate]
best_trend_specification = ADVANCED_SPEC_BY_NAME[best_trend_candidate]

advanced_raw_final = fit_final_advanced_candidate(
    best_raw_specification,
    advanced_full_train,
    advanced_ordered_test,
    candidate_iterations(best_raw_candidate),
)
advanced_trend_final = fit_final_advanced_candidate(
    best_trend_specification,
    advanced_full_train,
    advanced_ordered_test,
    candidate_iterations(best_trend_candidate),
)

advanced_blend_prediction = (
    ADVANCED_RAW_BLEND_WEIGHT * advanced_raw_final["prediction"]
    + (1 - ADVANCED_RAW_BLEND_WEIGHT) * advanced_trend_final["prediction"]
)

if ADVANCED_SELECTED_MODEL == "optimized_raw_trend_blend":
    advanced_selected_prediction = advanced_blend_prediction
elif ADVANCED_SELECTED_MODEL == best_raw_candidate:
    advanced_selected_prediction = advanced_raw_final["prediction"]
elif ADVANCED_SELECTED_MODEL == best_trend_candidate:
    advanced_selected_prediction = advanced_trend_final["prediction"]
else:
    raise ValueError(
        f"Unexpected selected model: {ADVANCED_SELECTED_MODEL}"
    )


def make_advanced_submission(prediction):
    result = pd.DataFrame({
        "Id": advanced_ordered_test["ROW_ID"].astype(int),
        "Predicted": np.asarray(prediction, dtype=float),
    }).sort_values("Id").reset_index(drop=True)
    assert len(result) == 50_000
    assert result["Id"].equals(pd.Series(np.arange(50_000), name="Id"))
    assert result["Predicted"].notna().all()
    assert np.isfinite(result["Predicted"]).all()
    return result


advanced_raw_submission = make_advanced_submission(
    advanced_raw_final["prediction"]
)
advanced_trend_submission = make_advanced_submission(
    advanced_trend_final["prediction"]
)
advanced_blend_submission = make_advanced_submission(
    advanced_blend_prediction
)
advanced_selected_submission = make_advanced_submission(
    advanced_selected_prediction
)

advanced_raw_path = MODEL_ROOT / "submission_advanced_raw_or_log.csv"
advanced_trend_path = MODEL_ROOT / "submission_advanced_trend_residual.csv"
advanced_blend_path = MODEL_ROOT / "submission_advanced_blend.csv"
advanced_selected_path = MODEL_ROOT / "submission_advanced_selected.csv"
advanced_cv_path = MODEL_ROOT / "advanced_rolling_cv_results.csv"
advanced_summary_path = MODEL_ROOT / "advanced_model_summary.csv"
advanced_metadata_path = MODEL_ROOT / "advanced_model_metadata.json"
advanced_raw_model_path = MODEL_ROOT / "advanced_raw_or_log_model.cbm"
advanced_trend_model_path = MODEL_ROOT / "advanced_trend_residual_model.cbm"

advanced_raw_submission.to_csv(advanced_raw_path, index=False)
advanced_trend_submission.to_csv(advanced_trend_path, index=False)
advanced_blend_submission.to_csv(advanced_blend_path, index=False)
advanced_selected_submission.to_csv(advanced_selected_path, index=False)
advanced_cv_results.to_csv(advanced_cv_path, index=False)
advanced_summary.to_csv(advanced_summary_path, index=False)
advanced_raw_final["model"].save_model(advanced_raw_model_path)
advanced_trend_final["model"].save_model(advanced_trend_model_path)

advanced_metadata = {
    "full_search": bool(ADVANCED_FULL_SEARCH),
    "folds": int(ADVANCED_N_FOLDS),
    "forecast_horizon_months": int(ADVANCED_HORIZON_MONTHS),
    "best_raw_candidate": best_raw_candidate,
    "best_trend_candidate": best_trend_candidate,
    "raw_blend_weight": float(ADVANCED_RAW_BLEND_WEIGHT),
    "selected_model": ADVANCED_SELECTED_MODEL,
    "raw_iterations": int(advanced_raw_final["iterations"]),
    "trend_iterations": int(advanced_trend_final["iterations"]),
    "raw_smearing": float(advanced_raw_final["smearing"]),
    "trend_smearing": float(advanced_trend_final["smearing"]),
    "macro_used": False,
    "macro_reason": (
        "Only future-origin-safe macro forecasts should be admitted; realized "
        "test-period macro values are excluded."
    ),
    "segment_columns": ADVANCED_SEGMENT_COLUMNS,
    "outputs": {
        "selected_submission": str(advanced_selected_path),
        "raw_submission": str(advanced_raw_path),
        "trend_submission": str(advanced_trend_path),
        "blend_submission": str(advanced_blend_path),
    },
}
advanced_metadata_path.write_text(
    json.dumps(advanced_metadata, indent=2),
    encoding="utf-8",
)

print(json.dumps(advanced_metadata, indent=2))
display(advanced_selected_submission.head())
print(f"Selected advanced submission: {advanced_selected_path}")

### 11.5 Interpretation and safeguards

- Do not assume that the trend-residual model must win. It should be selected only
  if reconstructed-rent RMSE improves across multiple forward folds.
- A positive validation bias indicates that the market trend is still being
  extrapolated too aggressively. Prefer damped or last-level forecasts in that
  case.
- The state-space and seasonal methods have limited history available. Their
  worst-fold performance is therefore as important as their mean score.
- Recency weighting preserves 2021-2022 observations but reduces their influence.
  The chosen half-life is a model-selection result, not an economic fact.
- Hierarchical trends are deliberately restricted to town and flat type and are
  shrunk toward the common index. Block-level trend forecasting would be too
  sparse and unstable.
- The smearing correction is estimated inside each fold and again from the final
  labelled training data. It is never estimated from validation or test labels.
- Macro variables remain excluded until genuine as-of-cutoff forecasts or frozen
  vintages are available for the complete prediction horizon.
- The Kaggle public leaderboard should be treated as a secondary check. Repeatedly
  choosing models from its public score can overfit the public subset and may not
  transfer to the private leaderboard.

## 12. Leakage-safe comparable-rent features and CatBoost optimization

The temporal experiments above try to forecast one common market path. This
section instead gives CatBoost a local, historical reference for comparable
flats while retaining direct prediction of the raw rent target.

The new candidate features summarize earlier labelled transactions at five
progressively more specific levels:

1. Singapore-wide market;
2. flat type;
3. town and flat type;
4. town, flat type and flat model;
5. postal code or address and flat type.

Each level supplies a smoothed prior median, recent 12-month median, most
recent monthly median, transaction counts and months since the last
comparable transaction. Explicit categorical crosses are also included so
CatBoost can model interactions such as postal code by flat type.

This section is independent of the advanced trend section. It can be run
after the original data preparation and CatBoost helper cells even when the
statsmodels trend experiments are skipped. It writes new result files and
never overwrites the earlier submissions.


In [ ]:
HISTORY_CACHE_VERSION = "monthly_prior_v1"
HISTORY_RECENT_MONTHS = 12
HISTORY_SMOOTHING_STRENGTH = 12.0
HISTORY_DATE_COLUMN = "RENT_APPROVAL_DATE"

HISTORY_KEY_SPECS = [
    {"name": "GLOBAL", "columns": [], "parent": None},
    {"name": "FLAT_TYPE", "columns": ["FLAT_TYPE"], "parent": "GLOBAL"},
    {
        "name": "TOWN_FLAT",
        "columns": ["TOWN", "FLAT_TYPE"],
        "parent": "FLAT_TYPE",
    },
    {
        "name": "TOWN_FLAT_MODEL",
        "columns": ["TOWN", "FLAT_TYPE", "FLAT_MODEL"],
        "parent": "TOWN_FLAT",
    },
    {
        "name": "POSTAL_FLAT",
        "columns": ["POSTAL_CODE", "FLAT_TYPE"],
        "parent": "TOWN_FLAT",
    },
    {
        "name": "ADDRESS_FLAT",
        "columns": ["BLOCK", "STREET", "FLAT_TYPE"],
        "parent": "POSTAL_FLAT",
    },
]

HISTORY_CATEGORICAL_CROSSES = {
    "CMP_TOWN_FLAT": ["TOWN", "FLAT_TYPE"],
    "CMP_TOWN_FLAT_MODEL": ["TOWN", "FLAT_TYPE", "FLAT_MODEL"],
    "CMP_POSTAL_FLAT": ["POSTAL_CODE", "FLAT_TYPE"],
    "CMP_ADDRESS_FLAT": ["BLOCK", "STREET", "FLAT_TYPE"],
}


def make_composite_key(frame, columns):
    if not columns:
        return pd.Series("__ALL__", index=frame.index, dtype="string")
    key = frame[columns[0]].astype("string").fillna("__MISSING__")
    for column in columns[1:]:
        key = key.str.cat(
            frame[column].astype("string").fillna("__MISSING__"),
            sep=" | ",
        )
    return key


def add_comparable_keys(frame):
    out = frame.copy()
    for specification in HISTORY_KEY_SPECS:
        out[f"__KEY_{specification['name']}"] = make_composite_key(
            out,
            specification["columns"],
        )
    for feature_name, columns in HISTORY_CATEGORICAL_CROSSES.items():
        out[feature_name] = make_composite_key(out, columns)
    return out


HISTORY_KEY_COLUMNS = [
    f"__KEY_{specification['name']}"
    for specification in HISTORY_KEY_SPECS
]

display(pd.DataFrame(HISTORY_KEY_SPECS))


### 12.1 Leakage controls

Manual target-derived features are safe only when their construction matches
the forecast setup.

- A training row uses monthly aggregates from strictly earlier months. Its
  own rent and every rent from its approval month are excluded.
- A validation encoder is fitted only on that fold's fitting partition. It
  remains frozen throughout the complete validation horizon; validation
  labels never update later validation months.
- The final test encoder is fitted only on labelled training observations
  strictly before the first test month. Test predictions are never fed back
  as observed rents.
- Sparse local medians are shrunk toward progressively broader medians. The
  corresponding counts tell CatBoost how reliable each reference is.

Monthly medians are used as the elementary observations. This prevents a
block with many duplicate-looking transactions in one month from dominating
its long-run reference simply because it has more rows that month.


In [ ]:
def monthly_history_table(frame, key_column):
    return (
        frame.groupby(
            [key_column, HISTORY_DATE_COLUMN],
            as_index=False,
            observed=True,
        )
        .agg(
            MONTH_MEDIAN=(TARGET, "median"),
            MONTH_COUNT=(TARGET, "size"),
        )
        .sort_values([key_column, HISTORY_DATE_COLUMN])
        .reset_index(drop=True)
    )


def causal_raw_features_for_key(frame, key_column, prefix):
    monthly = monthly_history_table(frame, key_column)
    grouped = monthly.groupby(key_column, sort=False, observed=True)

    monthly[f"{prefix}_MEDIAN_ALL_RAW"] = grouped["MONTH_MEDIAN"].transform(
        lambda values: values.shift(1).expanding(min_periods=1).median()
    )
    monthly[f"{prefix}_COUNT_ALL"] = (
        grouped["MONTH_COUNT"].cumsum() - monthly["MONTH_COUNT"]
    )
    monthly[f"{prefix}_MEDIAN_12M_RAW"] = grouped[
        "MONTH_MEDIAN"
    ].transform(
        lambda values: values.shift(1).rolling(
            HISTORY_RECENT_MONTHS,
            min_periods=1,
        ).median()
    )
    monthly[f"{prefix}_COUNT_12M"] = grouped["MONTH_COUNT"].transform(
        lambda values: values.shift(1).rolling(
            HISTORY_RECENT_MONTHS,
            min_periods=1,
        ).sum()
    )
    monthly[f"{prefix}_LAST_MEDIAN_RAW"] = grouped[
        "MONTH_MEDIAN"
    ].shift(1)
    monthly[f"{prefix}_LAST_DATE"] = grouped[
        HISTORY_DATE_COLUMN
    ].shift(1)

    feature_columns = [
        column
        for column in monthly.columns
        if column.startswith(f"{prefix}_")
    ]
    return frame[["ROW_ID", HISTORY_DATE_COLUMN, key_column]].merge(
        monthly[
            [key_column, HISTORY_DATE_COLUMN, *feature_columns]
        ],
        on=[key_column, HISTORY_DATE_COLUMN],
        how="left",
        validate="many_to_one",
    ).drop(columns=[key_column, HISTORY_DATE_COLUMN])


def finalize_history_features(base_frame, raw_features):
    raw_features = (
        raw_features.set_index("ROW_ID")
        .reindex(base_frame["ROW_ID"])
        .reset_index()
    )
    result = pd.DataFrame({"ROW_ID": base_frame["ROW_ID"].to_numpy()})
    date_by_id = base_frame.set_index("ROW_ID")[HISTORY_DATE_COLUMN]

    for specification in HISTORY_KEY_SPECS:
        name = specification["name"]
        parent = specification["parent"]
        prefix = f"HIST_{name}"
        count_all = raw_features[f"{prefix}_COUNT_ALL"].fillna(0.0)
        count_recent = raw_features[f"{prefix}_COUNT_12M"].fillna(0.0)
        median_all_raw = raw_features[f"{prefix}_MEDIAN_ALL_RAW"]
        median_recent_raw = raw_features[f"{prefix}_MEDIAN_12M_RAW"]

        if parent is None:
            median_all = median_all_raw
            median_recent = median_recent_raw.fillna(median_all)
        else:
            parent_all = result[f"HIST_{parent}_MEDIAN_ALL"]
            parent_recent = result[f"HIST_{parent}_MEDIAN_12M"].fillna(
                parent_all
            )
            median_all = (
                count_all * median_all_raw.fillna(parent_all)
                + HISTORY_SMOOTHING_STRENGTH * parent_all
            ) / (count_all + HISTORY_SMOOTHING_STRENGTH)
            median_recent = (
                count_recent
                * median_recent_raw.fillna(parent_recent)
                + HISTORY_SMOOTHING_STRENGTH * parent_recent
            ) / (count_recent + HISTORY_SMOOTHING_STRENGTH)
            median_all = median_all.where(count_all.gt(0), parent_all)
            median_recent = median_recent.where(
                count_recent.gt(0),
                parent_recent,
            )

        last_median = raw_features[f"{prefix}_LAST_MEDIAN_RAW"].fillna(
            median_recent
        )
        last_dates = pd.to_datetime(
            raw_features[f"{prefix}_LAST_DATE"],
            errors="coerce",
        )
        row_dates = pd.to_datetime(
            raw_features["ROW_ID"].map(date_by_id),
            errors="coerce",
        )
        months_since = (
            (row_dates.dt.year - last_dates.dt.year) * 12
            + row_dates.dt.month
            - last_dates.dt.month
        ).clip(lower=0)

        result[f"{prefix}_MEDIAN_ALL"] = median_all.to_numpy(float)
        result[f"{prefix}_MEDIAN_12M"] = median_recent.to_numpy(float)
        result[f"{prefix}_LAST_MEDIAN"] = last_median.to_numpy(float)
        result[f"{prefix}_LOG_COUNT_ALL"] = np.log1p(
            count_all.to_numpy(float)
        )
        result[f"{prefix}_LOG_COUNT_12M"] = np.log1p(
            count_recent.to_numpy(float)
        )
        result[f"{prefix}_MONTHS_SINCE_LAST"] = months_since.to_numpy(
            float
        )

    return result


def make_causal_history_features(frame):
    base = add_comparable_keys(
        frame.sort_values([HISTORY_DATE_COLUMN, "ROW_ID"])
        .reset_index(drop=True)
    )
    raw = pd.DataFrame({"ROW_ID": base["ROW_ID"].to_numpy()})
    for specification in HISTORY_KEY_SPECS:
        name = specification["name"]
        block = causal_raw_features_for_key(
            base,
            f"__KEY_{name}",
            f"HIST_{name}",
        )
        raw = raw.merge(block, on="ROW_ID", how="left", validate="one_to_one")
    return finalize_history_features(base, raw)


def fit_comparable_history_encoder(history_frame, cutoff):
    cutoff = pd.Timestamp(cutoff)
    history = add_comparable_keys(history_frame)
    history = history.loc[history[HISTORY_DATE_COLUMN] < cutoff].copy()
    if history.empty:
        raise ValueError("Comparable-history encoder received no prior rows.")

    recent_start = cutoff - pd.DateOffset(months=HISTORY_RECENT_MONTHS)
    encoder = {"cutoff": cutoff, "specifications": {}}

    for specification in HISTORY_KEY_SPECS:
        name = specification["name"]
        key_column = f"__KEY_{name}"
        monthly = monthly_history_table(history, key_column)

        all_stats = (
            monthly.groupby(key_column, observed=True)
            .agg(
                MEDIAN_ALL_RAW=("MONTH_MEDIAN", "median"),
                COUNT_ALL=("MONTH_COUNT", "sum"),
            )
        )
        recent_stats = (
            monthly.loc[monthly[HISTORY_DATE_COLUMN] >= recent_start]
            .groupby(key_column, observed=True)
            .agg(
                MEDIAN_12M_RAW=("MONTH_MEDIAN", "median"),
                COUNT_12M=("MONTH_COUNT", "sum"),
            )
        )
        latest = (
            monthly.sort_values(HISTORY_DATE_COLUMN)
            .groupby(key_column, observed=True)
            .tail(1)
            .set_index(key_column)[[HISTORY_DATE_COLUMN, "MONTH_MEDIAN"]]
            .rename(
                columns={
                    HISTORY_DATE_COLUMN: "LAST_DATE",
                    "MONTH_MEDIAN": "LAST_MEDIAN_RAW",
                }
            )
        )
        encoder["specifications"][name] = (
            all_stats.join(recent_stats, how="left")
            .join(latest, how="left")
        )
    return encoder


def apply_comparable_history_encoder(score_frame, encoder):
    base = add_comparable_keys(score_frame)
    raw = pd.DataFrame({"ROW_ID": base["ROW_ID"].to_numpy()})

    for specification in HISTORY_KEY_SPECS:
        name = specification["name"]
        key_column = f"__KEY_{name}"
        prefix = f"HIST_{name}"
        statistics = encoder["specifications"][name]
        keys = base[key_column]

        raw[f"{prefix}_MEDIAN_ALL_RAW"] = keys.map(
            statistics["MEDIAN_ALL_RAW"]
        )
        raw[f"{prefix}_COUNT_ALL"] = keys.map(
            statistics["COUNT_ALL"]
        )
        raw[f"{prefix}_MEDIAN_12M_RAW"] = keys.map(
            statistics["MEDIAN_12M_RAW"]
        )
        raw[f"{prefix}_COUNT_12M"] = keys.map(
            statistics["COUNT_12M"]
        )
        raw[f"{prefix}_LAST_MEDIAN_RAW"] = keys.map(
            statistics["LAST_MEDIAN_RAW"]
        )
        raw[f"{prefix}_LAST_DATE"] = keys.map(statistics["LAST_DATE"])

    history_features = finalize_history_features(base, raw)
    out = base.merge(
        history_features,
        on="ROW_ID",
        how="left",
        validate="one_to_one",
    )
    return out.drop(columns=HISTORY_KEY_COLUMNS)


def attach_causal_history_features(frame, causal_feature_table):
    out = add_comparable_keys(frame).drop(columns=HISTORY_KEY_COLUMNS)
    return out.merge(
        causal_feature_table,
        on="ROW_ID",
        how="left",
        validate="one_to_one",
    )


### 12.2 Build and cache causal training features

The causal table is deterministic but involves grouped expanding medians.
It is cached inside the local model-output directory so it does not need to
be rebuilt every time the notebook kernel restarts. The cache fingerprint
includes the row count, date range and target sum; it is regenerated when
the labelled training table changes.


In [ ]:
HISTORY_CACHE_PATH = MODEL_ROOT / "comparable_history_train_features.parquet"
HISTORY_CACHE_METADATA_PATH = MODEL_ROOT / "comparable_history_cache_metadata.json"

history_cache_signature = {
    "version": HISTORY_CACHE_VERSION,
    "rows": int(len(train)),
    "first_month": str(train[HISTORY_DATE_COLUMN].min().date()),
    "last_month": str(train[HISTORY_DATE_COLUMN].max().date()),
    "target_sum": float(train[TARGET].sum()),
}

cache_is_current = False
if HISTORY_CACHE_PATH.exists() and HISTORY_CACHE_METADATA_PATH.exists():
    cached_metadata = json.loads(
        HISTORY_CACHE_METADATA_PATH.read_text(encoding="utf-8")
    )
    cache_is_current = cached_metadata == history_cache_signature

if cache_is_current:
    causal_history_features = pd.read_parquet(HISTORY_CACHE_PATH)
    print(f"Loaded comparable-history cache: {HISTORY_CACHE_PATH}")
else:
    causal_history_features = make_causal_history_features(train)
    causal_history_features.to_parquet(HISTORY_CACHE_PATH, index=False)
    HISTORY_CACHE_METADATA_PATH.write_text(
        json.dumps(history_cache_signature, indent=2),
        encoding="utf-8",
    )
    print(f"Built comparable-history cache: {HISTORY_CACHE_PATH}")

assert len(causal_history_features) == len(train)
assert causal_history_features["ROW_ID"].is_unique
assert set(causal_history_features["ROW_ID"]) == set(train["ROW_ID"])

history_feature_columns = [
    column
    for column in causal_history_features.columns
    if column != "ROW_ID"
]
print(f"Historical numeric features: {len(history_feature_columns)}")
display(causal_history_features[history_feature_columns].describe().T)


### 12.3 Forward validation and compact CatBoost search

The validation horizon is set to the number of distinct months in the
official test table, currently 17. This is deliberately longer than the
earlier 12-month holdout. Two expanding folds are used by default because
the labelled history is short; their overlap must be considered when
interpreting very small gains.

The reference candidate reruns raw CatBoost under exactly the same folds.
This makes the feature comparison fair. Two enhanced candidates test the
existing tree parameters and a shallower, more strongly regularized model
with second-order categorical combinations. Set `HISTORY_FULL_SEARCH=True`
to add two more parameter combinations and three validation folds.


In [ ]:
HISTORY_FULL_SEARCH = False
HISTORY_VALIDATION_MONTHS = int(test[HISTORY_DATE_COLUMN].nunique())
HISTORY_FOLD_STEP_MONTHS = 6
HISTORY_N_FOLDS = 3 if HISTORY_FULL_SEARCH else 2
HISTORY_ITERATIONS = 1_400 if HISTORY_FULL_SEARCH else 900
HISTORY_EARLY_STOPPING = 100


def make_comparable_history_folds(
    frame,
    horizon_months,
    n_folds,
    step_months,
    minimum_training_months=24,
):
    months = pd.DatetimeIndex(sorted(frame[HISTORY_DATE_COLUMN].unique()))
    final_start = len(months) - horizon_months
    starts = [
        final_start - step_months * offset
        for offset in reversed(range(n_folds))
    ]
    if min(starts) < minimum_training_months:
        raise ValueError(
            "Insufficient labelled history for the requested comparable-"
            "history folds. Reduce HISTORY_N_FOLDS or the horizon."
        )

    folds = []
    for fold_number, start in enumerate(starts, start=1):
        validation_months = months[start:start + horizon_months]
        fit_part = frame.loc[
            frame[HISTORY_DATE_COLUMN] < validation_months[0]
        ].sort_values([HISTORY_DATE_COLUMN, "ROW_ID"]).reset_index(drop=True)
        valid_part = frame.loc[
            frame[HISTORY_DATE_COLUMN].isin(validation_months)
        ].sort_values([HISTORY_DATE_COLUMN, "ROW_ID"]).reset_index(drop=True)
        assert (
            fit_part[HISTORY_DATE_COLUMN].max()
            < valid_part[HISTORY_DATE_COLUMN].min()
        )
        folds.append(
            {
                "fold": fold_number,
                "fit": fit_part,
                "valid": valid_part,
                "fit_end": fit_part[HISTORY_DATE_COLUMN].max(),
                "validation_start": valid_part[HISTORY_DATE_COLUMN].min(),
                "validation_end": valid_part[HISTORY_DATE_COLUMN].max(),
            }
        )
    return folds


history_folds = make_comparable_history_folds(
    train,
    horizon_months=HISTORY_VALIDATION_MONTHS,
    n_folds=HISTORY_N_FOLDS,
    step_months=HISTORY_FOLD_STEP_MONTHS,
)

HISTORY_CANDIDATES = [
    {
        "name": "raw_reference",
        "use_history": False,
        "parameters": {},
    },
    {
        "name": "history_current_parameters",
        "use_history": True,
        "parameters": {},
    },
    {
        "name": "history_shallow_regularized",
        "use_history": True,
        "parameters": {
            "depth": 7,
            "l2_leaf_reg": 12.0,
            "random_strength": 0.25,
            "bagging_temperature": 0.75,
            "max_ctr_complexity": 2,
        },
    },
]

if HISTORY_FULL_SEARCH:
    HISTORY_CANDIDATES.extend(
        [
            {
                "name": "history_depth_6_regularized",
                "use_history": True,
                "parameters": {
                    "depth": 6,
                    "learning_rate": 0.04,
                    "l2_leaf_reg": 18.0,
                    "random_strength": 0.1,
                    "bagging_temperature": 1.0,
                    "max_ctr_complexity": 2,
                },
            },
            {
                "name": "history_depth_9",
                "use_history": True,
                "parameters": {
                    "depth": 9,
                    "learning_rate": 0.04,
                    "l2_leaf_reg": 15.0,
                    "random_strength": 0.5,
                    "bagging_temperature": 0.5,
                    "max_ctr_complexity": 2,
                },
            },
        ]
    )


def comparable_catboost_parameters(overrides=None, random_seed=RANDOM_SEED):
    parameters = {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "iterations": HISTORY_ITERATIONS,
        "learning_rate": 0.05,
        "depth": 8,
        "l2_leaf_reg": 8.0,
        "random_strength": 0.5,
        "bagging_temperature": 0.5,
        "random_seed": int(random_seed),
        "has_time": True,
        "allow_writing_files": False,
        "thread_count": -1,
        "verbose": False,
    }
    if overrides:
        parameters.update(overrides)
    return parameters


def prepare_history_fold_frames(fold, use_history):
    if not use_history:
        return fold["fit"].copy(), fold["valid"].copy()

    fit_enriched = attach_causal_history_features(
        fold["fit"],
        causal_history_features,
    )
    encoder = fit_comparable_history_encoder(
        fold["fit"],
        cutoff=fold["validation_start"],
    )
    valid_enriched = apply_comparable_history_encoder(
        fold["valid"],
        encoder,
    )
    return fit_enriched, valid_enriched


def fit_history_candidate(candidate, fold):
    fit_frame, valid_frame = prepare_history_fold_frames(
        fold,
        candidate["use_history"],
    )
    X_fit, categorical_columns = make_feature_frame(
        fit_frame,
        include_macro=False,
    )
    X_valid, valid_categorical_columns = make_feature_frame(
        valid_frame,
        include_macro=False,
    )
    assert list(X_fit.columns) == list(X_valid.columns)
    assert categorical_columns == valid_categorical_columns

    model = CatBoostRegressor(
        **comparable_catboost_parameters(candidate["parameters"])
    )
    model.fit(
        Pool(
            X_fit,
            label=fit_frame[TARGET],
            cat_features=categorical_columns,
        ),
        eval_set=Pool(
            X_valid,
            label=valid_frame[TARGET],
            cat_features=categorical_columns,
        ),
        use_best_model=True,
        early_stopping_rounds=HISTORY_EARLY_STOPPING,
    )
    prediction = model.predict(X_valid)
    return {
        "model": model,
        "prediction": prediction,
        "metrics": regression_metrics(valid_frame[TARGET], prediction),
        "best_iteration": max(int(model.get_best_iteration() + 1), 50),
    }


display(
    pd.DataFrame(
        [
            {
                "fold": fold["fold"],
                "fit_rows": len(fold["fit"]),
                "fit_end": fold["fit_end"],
                "validation_rows": len(fold["valid"]),
                "validation_start": fold["validation_start"],
                "validation_end": fold["validation_end"],
            }
            for fold in history_folds
        ]
    )
)
display(pd.DataFrame(HISTORY_CANDIDATES))


In [ ]:
history_cv_rows = []
history_validation_predictions = []

for fold in history_folds:
    print(
        f"Fold {fold['fold']}: fit through {fold['fit_end'].date()}, "
        f"validate {fold['validation_start'].date()} "
        f"to {fold['validation_end'].date()}"
    )
    for candidate in HISTORY_CANDIDATES:
        outcome = fit_history_candidate(candidate, fold)
        history_cv_rows.append(
            {
                "fold": fold["fold"],
                "fit_end": fold["fit_end"],
                "validation_start": fold["validation_start"],
                "validation_end": fold["validation_end"],
                "candidate": candidate["name"],
                "use_history": candidate["use_history"],
                "best_iteration": outcome["best_iteration"],
                **outcome["metrics"],
            }
        )
        prediction_frame = fold["valid"][
            ["ROW_ID", HISTORY_DATE_COLUMN, TARGET]
        ].copy()
        prediction_frame["fold"] = fold["fold"]
        prediction_frame["candidate"] = candidate["name"]
        prediction_frame["PREDICTION"] = outcome["prediction"]
        history_validation_predictions.append(prediction_frame)
        print(
            f"  {candidate['name']}: "
            f"RMSE={outcome['metrics']['RMSE']:,.2f}, "
            f"MAE={outcome['metrics']['MAE']:,.2f}, "
            f"bias={outcome['metrics']['BIAS']:,.2f}"
        )

history_cv_results = pd.DataFrame(history_cv_rows)
history_oof_predictions = pd.concat(
    history_validation_predictions,
    ignore_index=True,
)
history_model_summary = (
    history_cv_results.groupby(
        ["candidate", "use_history"],
        as_index=False,
    )
    .agg(
        mean_RMSE=("RMSE", "mean"),
        std_RMSE=("RMSE", "std"),
        worst_RMSE=("RMSE", "max"),
        mean_MAE=("MAE", "mean"),
        mean_R2=("R2", "mean"),
        mean_BIAS=("BIAS", "mean"),
        median_best_iteration=("best_iteration", "median"),
    )
    .sort_values(["mean_RMSE", "worst_RMSE"])
    .reset_index(drop=True)
)

reference_rmse = history_model_summary.loc[
    history_model_summary["candidate"].eq("raw_reference"),
    "mean_RMSE",
].iloc[0]
history_model_summary["RMSE_GAIN_VS_REFERENCE"] = (
    reference_rmse - history_model_summary["mean_RMSE"]
)
HISTORY_SELECTED_CANDIDATE = history_model_summary.iloc[0]["candidate"]

print(f"Selected candidate: {HISTORY_SELECTED_CANDIDATE}")
display(history_model_summary)

history_cv_path = MODEL_ROOT / "comparable_history_cv_results.csv"
history_summary_path = MODEL_ROOT / "comparable_history_model_summary.csv"
history_cv_results.to_csv(history_cv_path, index=False)
history_model_summary.to_csv(history_summary_path, index=False)
print(f"Saved: {history_cv_path}")
print(f"Saved: {history_summary_path}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_summary = history_model_summary.sort_values("mean_RMSE")
axes[0].barh(
    plot_summary["candidate"],
    plot_summary["mean_RMSE"],
    xerr=plot_summary["std_RMSE"].fillna(0),
    color=np.where(plot_summary["use_history"], "#59A14F", "#4C78A8"),
    alpha=0.9,
)
axes[0].invert_yaxis()
axes[0].set_xlabel("Mean forward-validation RMSE (SGD)")
axes[0].set_title("Comparable-history CatBoost candidates")
axes[0].grid(axis="x", alpha=0.25)

latest_fold = history_cv_results["fold"].max()
latest_predictions = history_oof_predictions.loc[
    history_oof_predictions["fold"].eq(latest_fold)
]
monthly_actual = (
    latest_predictions[[HISTORY_DATE_COLUMN, TARGET]]
    .drop_duplicates()
    .groupby(HISTORY_DATE_COLUMN, as_index=False)[TARGET]
    .mean()
)
# Reconstruct actual monthly means without duplicate candidate rows.
monthly_actual = (
    history_folds[-1]["valid"]
    .groupby(HISTORY_DATE_COLUMN, as_index=False)[TARGET]
    .mean()
)
axes[1].plot(
    monthly_actual[HISTORY_DATE_COLUMN],
    monthly_actual[TARGET],
    color="black",
    linewidth=3,
    marker="o",
    label="Actual",
)
for candidate_name, group in latest_predictions.groupby("candidate"):
    monthly_prediction = (
        group.groupby(HISTORY_DATE_COLUMN, as_index=False)["PREDICTION"]
        .mean()
    )
    axes[1].plot(
        monthly_prediction[HISTORY_DATE_COLUMN],
        monthly_prediction["PREDICTION"],
        marker=".",
        linewidth=1.7,
        label=candidate_name,
    )
axes[1].set_title("Latest fold: monthly actual versus predicted")
axes[1].set_ylabel("Mean monthly rent (SGD)")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()


### 12.4 Final refit, small seed ensemble and Kaggle submission

The winning candidate is refitted on all labelled rows. If it uses historical
features, each labelled row retains its strictly prior-month encoding. The
test encoder is frozen immediately before the first test month, so no test
label or prediction can update the comparable-rent statistics.

Three random seeds are averaged. This is a small variance-reduction ensemble,
not a temporal blend. Set `HISTORY_ENSEMBLE_SEEDS` to a single value when a
faster run is required.


In [ ]:
HISTORY_ENSEMBLE_SEEDS = [42, 314, 2026]

selected_history_specification = next(
    candidate
    for candidate in HISTORY_CANDIDATES
    if candidate["name"] == HISTORY_SELECTED_CANDIDATE
)
selected_iteration_rows = history_cv_results.loc[
    history_cv_results["candidate"].eq(HISTORY_SELECTED_CANDIDATE),
    "best_iteration",
]
selected_iterations = max(
    int(np.median(selected_iteration_rows)),
    50,
)

if selected_history_specification["use_history"]:
    final_train_frame = attach_causal_history_features(
        train,
        causal_history_features,
    )
    final_history_cutoff = pd.Timestamp(test[HISTORY_DATE_COLUMN].min())
    final_encoder = fit_comparable_history_encoder(
        train,
        cutoff=final_history_cutoff,
    )
    final_test_frame = apply_comparable_history_encoder(test, final_encoder)
else:
    final_train_frame = train.copy()
    final_test_frame = test.copy()
    final_history_cutoff = None

final_train_frame = final_train_frame.sort_values(
    [HISTORY_DATE_COLUMN, "ROW_ID"]
).reset_index(drop=True)
final_test_frame = final_test_frame.sort_values("ROW_ID").reset_index(drop=True)

X_history_train, history_categorical_columns = make_feature_frame(
    final_train_frame,
    include_macro=False,
)
X_history_test, history_test_categorical_columns = make_feature_frame(
    final_test_frame,
    include_macro=False,
)
assert list(X_history_train.columns) == list(X_history_test.columns)
assert history_categorical_columns == history_test_categorical_columns

history_seed_predictions = []
history_final_models = []

for seed in HISTORY_ENSEMBLE_SEEDS:
    final_parameters = comparable_catboost_parameters(
        selected_history_specification["parameters"],
        random_seed=seed,
    )
    final_parameters["iterations"] = selected_iterations
    final_parameters["verbose"] = 200
    model = CatBoostRegressor(**final_parameters)
    model.fit(
        Pool(
            X_history_train,
            label=final_train_frame[TARGET],
            cat_features=history_categorical_columns,
        )
    )
    history_seed_predictions.append(model.predict(X_history_test))
    history_final_models.append(model)
    model_path = MODEL_ROOT / f"catboost_comparable_history_seed_{seed}.cbm"
    model.save_model(model_path)
    print(f"Saved: {model_path}")

history_test_prediction = np.mean(
    np.vstack(history_seed_predictions),
    axis=0,
)
history_submission = pd.DataFrame(
    {
        "Id": final_test_frame["ROW_ID"].astype(int),
        "Predicted": history_test_prediction.astype(float),
    }
).sort_values("Id").reset_index(drop=True)

assert len(history_submission) == len(test) == 50_000
assert history_submission["Id"].equals(
    pd.Series(np.arange(50_000), name="Id")
)
assert history_submission["Predicted"].notna().all()
assert np.isfinite(history_submission["Predicted"]).all()

history_submission_path = (
    MODEL_ROOT / "submission_catboost_comparable_history.csv"
)
history_metadata_path = MODEL_ROOT / "comparable_history_model_metadata.json"
history_importance_path = MODEL_ROOT / "comparable_history_feature_importance.csv"

history_submission.to_csv(history_submission_path, index=False)

history_importance = pd.DataFrame(
    {
        "feature": X_history_train.columns,
        "importance": np.mean(
            np.vstack(
                [
                    model.get_feature_importance()
                    for model in history_final_models
                ]
            ),
            axis=0,
        ),
    }
).sort_values("importance", ascending=False)
history_importance.to_csv(history_importance_path, index=False)

history_metadata = {
    "selected_candidate": HISTORY_SELECTED_CANDIDATE,
    "uses_comparable_history": bool(
        selected_history_specification["use_history"]
    ),
    "iterations": int(selected_iterations),
    "ensemble_seeds": [int(seed) for seed in HISTORY_ENSEMBLE_SEEDS],
    "validation_horizon_months": int(HISTORY_VALIDATION_MONTHS),
    "validation_folds": int(HISTORY_N_FOLDS),
    "test_history_cutoff_exclusive": (
        str(final_history_cutoff.date())
        if final_history_cutoff is not None
        else None
    ),
    "leakage_rule": (
        "Training encodings use strictly earlier months; validation and "
        "test encoders use labelled fitting rows only and remain frozen."
    ),
    "historical_numeric_feature_count": int(len(history_feature_columns)),
    "categorical_crosses": list(HISTORY_CATEGORICAL_CROSSES),
    "feature_count": int(X_history_train.shape[1]),
    "submission": str(history_submission_path),
}
history_metadata_path.write_text(
    json.dumps(history_metadata, indent=2),
    encoding="utf-8",
)

print(json.dumps(history_metadata, indent=2))
print(f"Kaggle submission: {history_submission_path}")
display(history_submission.head(10))
display(history_importance.head(30))


### 12.5 How to decide whether to use the enhanced submission

Compare `RMSE_GAIN_VS_REFERENCE` across every forward fold. A gain of only
one or two dollars is not convincing because the folds overlap and model
selection itself introduces noise. Prefer the enhanced model only when its
mean and worst-fold RMSE both improve and its monthly prediction path does
not develop a large systematic bias.

The most important leakage audit is simple: changing a validation or test
label must be unable to change any corresponding comparable-history input.
The implementation enforces this by fitting a frozen encoder from the fold's
labelled fitting rows before transforming the complete forecast window.

The new output is `submission_catboost_comparable_history.csv`. Keep the
original raw CatBoost submission as the benchmark and use Kaggle only as a
secondary check after the forward-validation comparison is complete.
